In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q datasets

from datasets import load_dataset, get_dataset_split_names
import os

out_dir = "/content/drive/MyDrive/RAG-project/data/t2-ragbench"
os.makedirs(out_dir, exist_ok=True)

for name in ["FinQA", "ConvFinQA", "TAT-DQA"]:
    splits = get_dataset_split_names("G4KMU/t2-ragbench", name)
    print(name, "splits:", splits)
    for split in splits:
        ds = load_dataset("G4KMU/t2-ragbench", name, split=split)
        ds.to_parquet(f"{out_dir}/{name}_{split}.parquet")
        print(f"  {split}: {len(ds)} rows saved")

In [ ]:
import pandas as pd

files = {
    "FinQA": ["train", "dev", "test"],
    "ConvFinQA": ["turn_0"],
    "TAT-DQA": ["train", "dev", "test"],
}

for name, splits in files.items():
    for split in splits:
        df = pd.read_parquet(f"{out_dir}/{name}_{split}.parquet")
        print(f"{name}/{split}: {len(df)} строк, {df['context_id'].nunique()} уникальных context_id")

In [ ]:
import re

def answer_findable(row):
    if pd.notna(row.get("table", None)):
        text = f"{row.get('table','')} {row.get('pre_text','')} {row.get('post_text','')}"
    else:
        text = row.get("context", "")
    nums = re.findall(r"-?\d[\d,]*\.?\d*", str(text).replace(",", ""))
    nums = [float(n) for n in nums if n not in ("", "-", ".")]
    try:
        ans = float(row["program_answer"])
    except (ValueError, TypeError):
        return None
    return any(abs(n - ans) < 0.01 for n in nums)

df_finqa = pd.read_parquet(f"{out_dir}/FinQA_train.parquet")
df_tatdqa = pd.read_parquet(f"{out_dir}/TAT-DQA_train.parquet")

for name, df in [("FinQA", df_finqa), ("TAT-DQA", df_tatdqa)]:
    sample = df.sample(200, random_state=42)
    results = sample.apply(answer_findable, axis=1)
    hits = results.sum()
    checked = results.notna().sum()
    print(f"{name}: {hits}/{checked} ответов найдены как число в контексте ({100*hits/checked:.1f}%)")

In [ ]:
for name in ["FinQA", "TAT-DQA"]:
    df = pd.read_parquet(f"{out_dir}/{name}_train.parquet")
    col = "table" if "table" in df.columns else "context"
    longest = df.loc[df[col].astype(str).str.len().nlargest(3).index]
    for _, row in longest.iterrows():
        print(f"--- {name} id={row['id']} len={len(str(row[col]))} ---")
        print(row[col])
        print()

In [ ]:
import re

def dup_ratio(text, min_block_len=80):
    parts = [p.strip() for p in re.split(r'\n{1,}|(?<=\.)\s{2,}', str(text)) if len(p.strip()) >= min_block_len]
    if not parts:
        return 0.0
    return 1 - len(set(parts)) / len(parts)

df = pd.read_parquet(f"{out_dir}/TAT-DQA_train.parquet")
sample = df.sample(300, random_state=42)
sample["dup_ratio"] = sample["context"].apply(dup_ratio)
print(sample["dup_ratio"].describe())
print("Доля строк с dup_ratio > 0.1:", (sample["dup_ratio"] > 0.1).me

In [ ]:
!pip install -q pymongo voyageai sentence-transformers pandas pyarrow

In [ ]:
MONGODB_URI = "YOU_KEY_HERE"
VOYAGE_API_KEY = "YOU_KEY_HERE"

print("MONGODB_URI загружен:", bool(MONGODB_URI))
print("VOYAGE_API_KEY загружен:", bool(VOYAGE_API_KEY))

In [ ]:
import pandas as pd

out_dir = "/content/drive/MyDrive/RAG-project/data/t2-ragbench"

def load_subset(name, splits):
    dfs = [pd.read_parquet(f"{out_dir}/{name}_{s}.parquet") for s in splits]
    return pd.concat(dfs, ignore_index=True)

finqa = load_subset("FinQA", ["train", "dev", "test"])
convfinqa = load_subset("ConvFinQA", ["turn_0"])
tatdqa = load_subset("TAT-DQA", ["train", "dev", "test"])

def sample_docs(df, n_docs, seed=42):
    unique_ctx = df["context_id"].drop_duplicates().sample(n_docs, random_state=seed)
    docs = df[df["context_id"].isin(unique_ctx)].drop_duplicates("context_id")
    queries = df[df["context_id"].isin(unique_ctx)]
    return docs, queries

docs_fq, q_fq = sample_docs(finqa, 24)
docs_cf, q_cf = sample_docs(convfinqa, 16)
docs_td, q_td = sample_docs(tatdqa, 20)

docs = pd.concat([docs_fq, docs_cf, docs_td], ignore_index=True)
queries = pd.concat([q_fq, q_cf, q_td], ignore_index=True)

print(f"Документов: {len(docs)}, вопросов к ним: {len(queries)}")

In [ ]:
import voyageai
import numpy as np
import time

vo = voyageai.Client(api_key=VOYAGE_API_KEY)

def embed_voyage(texts, input_type, batch_size=32, model="voyage-4"):
    vecs = []
    t0 = time.time()
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        result = vo.embed(batch, model=model, input_type=input_type)
        vecs.extend(result.embeddings)
    print(f"Voyage embed {len(texts)} текстов: {time.time()-t0:.1f} сек")
    return np.array(vecs)

doc_texts = docs["context"].tolist()
query_texts = queries["question"].tolist()

doc_emb_voyage = embed_voyage(doc_texts, input_type="document")
query_emb_voyage = embed_voyage(query_texts, input_type="query")

print("Размерность эмбеддинга:", doc_emb_voyage.shape)

In [ ]:
import voyageai
import numpy as np
import time

vo = voyageai.Client(api_key=VOYAGE_API_KEY)

def embed_voyage(texts, input_type, batch_size=5, model="voyage-4", delay=21):
    vecs = []
    t0 = time.time()
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        while True:
            try:
                result = vo.embed(batch, model=model, input_type=input_type)
                break
            except Exception as e:
                if "RateLimit" in type(e).__name__:
                    print(f"  Rate limit, жду 30 сек... ({i}/{len(texts)})")
                    time.sleep(30)
                else:
                    raise
        vecs.extend(result.embeddings)
        print(f"  {min(i+batch_size, len(texts))}/{len(texts)} готово")
        time.sleep(delay)
    print(f"Voyage embed {len(texts)} текстов: {time.time()-t0:.1f} сек")
    return np.array(vecs)

doc_texts = docs["context"].tolist()
query_texts = queries["question"].tolist()

doc_emb_voyage = embed_voyage(doc_texts, input_type="document")
query_emb_voyage = embed_voyage(query_texts, input_type="query")

print("Размерность эмбеддинга:", doc_emb_voyage.shape)

In [ ]:
from sentence_transformers import SentenceTransformer
import time

bge = SentenceTransformer("BAAI/bge-m3", device="cpu")

t0 = time.time()
doc_emb_bge = bge.encode(doc_texts, batch_size=16, show_progress_bar=True, normalize_embeddings=True)
query_emb_bge = bge.encode(query_texts, batch_size=16, show_progress_bar=True, normalize_embeddings=True)
print(f"BGE-M3 embed {len(doc_texts)+len(query_texts)} текстов: {time.time()-t0:.1f} сек")
print("Размерность эмбеддинга:", doc_emb_bge.shape)

In [ ]:
def recall_at_k(doc_emb, query_emb, docs_df, queries_df, k=5):
    doc_ctx_ids = docs_df["context_id"].tolist()
    doc_emb_norm = doc_emb / np.linalg.norm(doc_emb, axis=1, keepdims=True)
    query_emb_norm = query_emb / np.linalg.norm(query_emb, axis=1, keepdims=True)
    sims = query_emb_norm @ doc_emb_norm.T
    hits = 0
    for i, row in enumerate(queries_df.itertuples()):
        top_k_idx = np.argsort(-sims[i])[:k]
        retrieved_ctx = {doc_ctx_ids[j] for j in top_k_idx}
        if row.context_id in retrieved_ctx:
            hits += 1
    return hits / len(queries_df)

recall_voyage = recall_at_k(doc_emb_voyage, query_emb_voyage, docs, queries)
recall_bge = recall_at_k(doc_emb_bge, query_emb_bge, docs, queries)

print(f"Recall@5 Voyage AI (voyage-4): {recall_voyage:.3f}")
print(f"Recall@5 BGE-M3:               {recall_bge:.3f}")
print(f"Оценка стоимости Voyage AI на весь корпус: ~${6.5 * 0.02:.2f}")

In [ ]:
from pymongo import MongoClient
from pymongo.operations import SearchIndexModel
import time

client = MongoClient(MONGODB_URI)
db = client["rag_project"]
coll = db["t2_ragbench_test"]
coll.delete_many({})

records = []
for i, row in docs.reset_index(drop=True).iterrows():
    records.append({
        "context_id": row["context_id"],
        "context": row["context"],
        "company_name": row.get("company_name", ""),
        "embedding_voyage": doc_emb_voyage[i].tolist(),
    })
coll.insert_many(records)
print(f"Записано {len(records)} документов в Atlas")

vector_index_model = SearchIndexModel(
    definition={
        "fields": [
            {"type": "vector", "path": "embedding_voyage", "numDimensions": len(doc_emb_voyage[0]), "similarity": "cosine"},
        ]
    },
    name="vector_index_voyage",
    type="vectorSearch",
)
text_index_model = SearchIndexModel(
    definition={"mappings": {"dynamic": False, "fields": {"context": {"type": "string"}}}},
    name="text_index",
    type="search",
)

t0 = time.time()
coll.create_search_index(vector_index_model)
coll.create_search_index(text_index_model)

def wait_index_ready(coll, name, timeout=180):
    start = time.time()
    while time.time() - start < timeout:
        idx = list(coll.list_search_indexes(name))
        if idx and idx[0].get("queryable"):
            return True
        time.sleep(5)
    return False

ready_v = wait_index_ready(coll, "vector_index_voyage")
ready_t = wait_index_ready(coll, "text_index")
print(f"Индексы готовы за {time.time()-t0:.1f} сек (vector={ready_v}, text={ready_t})")

In [ ]:
test_query = queries.iloc[0]["question"]
test_query_emb = vo.embed([test_query], model="voyage-4", input_type="query").embeddings[0]

pipeline = [
    {
        "$rankFusion": {
            "input": {
                "pipelines": {
                    "vectorPipeline": [
                        {
                            "$vectorSearch": {
                                "index": "vector_index_voyage",
                                "path": "embedding_voyage",
                                "queryVector": test_query_emb,
                                "numCandidates": 50,
                                "limit": 10,
                            }
                        }
                    ],
                    "fullTextPipeline": [
                        {"$search": {"index": "text_index", "text": {"query": test_query, "path": "context"}}},
                        {"$limit": 10},
                    ],
                }
            },
            "combination": {"weights": {"vectorPipeline": 0.5, "fullTextPipeline": 0.5}},
        }
    },
    {"$limit": 5},
    {"$project": {"context_id": 1, "company_name": 1, "_id": 0}},
]

import time
t0 = time.time()
results = list(coll.aggregate(pipeline))
print(f"$rankFusion запрос выполнен за {time.time()-t0:.2f} сек")
print(f"Вопрос: {test_query}")
print(f"Правильный context_id: {queries.iloc[0]['context_id']}")
print("Топ-5 результатов:", results)

In [ ]:
ython
import time

t0 = time.time()
for i in range(5):
    vo.embed(["тестовый запрос"], model="voyage-4", input_type="query")
    print(f"Запрос {i+1} прошёл, {time.time()-t0:.1f} сек")

In [ ]:
ython
import time

t0 = time.time()
for i in range(5):
    vo.embed(["тестовый запрос"], model="voyage-4", input_type="query")
    print(f"Запрос {i+1} прошёл, {time.time()-t0:.1f} сек")

In [ ]:
import time

t0 = time.time()
for i in range(5):
    vo.embed(["тестовый запрос"], model="voyage-4", input_type="query")
    print(f"Запрос {i+1} прошёл, {time.time()-t0:.1f} сек")

In [ ]:
import time

t0 = time.time()
for i in range(5):
    vo.embed(["тестовый запрос"], model="voyage-4", input_type="query")
    print(f"Запрос {i+1} прошёл, {time.time()-t0:.1f} сек")

In [ ]:
import time

t0 = time.time()
for i in range(5):
    vo.embed(["тестовый запрос"], model="voyage-4", input_type="query")
    print(f"Запрос {i+1} прошёл, {time.time()-t0:.1f} сек")

In [ ]:
import time

t0 = time.time()
for i in range(5):
    vo.embed(["тестовый запрос"], model="voyage-4", input_type="query")
    print(f"Запрос {i+1} прошёл, {time.time()-t0:.1f} сек")

In [ ]:
import time

t0 = time.time()
for i in range(5):
    vo.embed(["тестовый запрос"], model="voyage-4", input_type="query")
    print(f"Запрос {i+1} прошёл, {time.time()-t0:.1f} сек")

In [ ]:
import time

t0 = time.time()
for i in range(5):
    vo.embed(["тестовый запрос"], model="voyage-4", input_type="query")
    print(f"Запрос {i+1} прошёл, {time.time()-t0:.1f} сек")

In [ ]:
import voyageai
import time

vo = voyageai.Client(api_key=VOYAGE_API_KEY)

t0 = time.time()
for i in range(5):
    vo.embed(["тестовый запрос"], model="voyage-4", input_type="query")
    print(f"Запрос {i+1} прошёл, {time.time()-t0:.1f} сек")

In [ ]:
!pip install -q voyageai

import voyageai
import time

VOYAGE_API_KEY = "YOU_KEY_HERE"
vo = voyageai.Client(api_key=VOYAGE_API_KEY)

t0 = time.time()
for i in range(5):
    vo.embed(["тестовый запрос"], model="voyage-4", input_type="query")
    print(f"Запрос {i+1} прошёл, {time.time()-t0:.1f} сек")

In [ ]:
!pip install -q voyageai

import voyageai
import time

VOYAGE_API_KEY = "YOU_KEY_HERE"
vo = voyageai.Client(api_key=VOYAGE_API_KEY)

t0 = time.time()
for i in range(5):
    vo.embed(["тестовый запрос"], model="voyage-4", input_type="query")
    print(f"Запрос {i+1} прошёл, {time.time()-t0:.1f} сек")

In [ ]:
import pandas as pd

out_dir = "/content/drive/MyDrive/RAG-project/data/t2-ragbench"

def load_subset(name, splits):
    dfs = [pd.read_parquet(f"{out_dir}/{name}_{s}.parquet") for s in splits]
    return pd.concat(dfs, ignore_index=True)

finqa = load_subset("FinQA", ["train", "dev", "test"])
convfinqa = load_subset("ConvFinQA", ["turn_0"])
tatdqa = load_subset("TAT-DQA", ["train", "dev", "test"])

all_qa = pd.concat([finqa, convfinqa, tatdqa], ignore_index=True)
all_docs = all_qa.drop_duplicates("context_id").reset_index(drop=True)

print(f"Всего вопросов: {len(all_qa)}, уникальных документов: {len(all_docs)}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q pymongo voyageai pandas pyarrow

import pandas as pd
import voyageai
import time

VOYAGE_API_KEY = "YOU_KEY_HERE"
MONGODB_URI = "YOU_KEY_HERE"

vo = voyageai.Client(api_key=VOYAGE_API_KEY)

out_dir = "/content/drive/MyDrive/RAG-project/data/t2-ragbench"
print("Готово: диск примонтирован, пакеты установлены, клиенты созданы.")

In [ ]:
def load_subset(name, splits):
    dfs = [pd.read_parquet(f"{out_dir}/{name}_{s}.parquet") for s in splits]
    return pd.concat(dfs, ignore_index=True)

finqa = load_subset("FinQA", ["train", "dev", "test"])
convfinqa = load_subset("ConvFinQA", ["turn_0"])
tatdqa = load_subset("TAT-DQA", ["train", "dev", "test"])

all_qa = pd.concat([finqa, convfinqa, tatdqa], ignore_index=True)
all_docs = all_qa.drop_duplicates("context_id").reset_index(drop=True)

print(f"Всего вопросов: {len(all_qa)}, уникальных документов: {len(all_docs)}")

In [ ]:
import numpy as np

def embed_voyage_full(texts, input_type, batch_size=64, model="voyage-4"):
    vecs = []
    t0 = time.time()
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        for attempt in range(3):
            try:
                result = vo.embed(batch, model=model, input_type=input_type)
                break
            except Exception as e:
                if "RateLimit" in type(e).__name__:
                    print(f"  Rate limit, жду 15 сек... ({i}/{len(texts)})")
                    time.sleep(15)
                else:
                    raise
        vecs.extend(result.embeddings)
        if (i // batch_size) % 10 == 0:
            print(f"  {min(i+batch_size, len(texts))}/{len(texts)} готово, {time.time()-t0:.0f} сек")
    print(f"Voyage embed {len(texts)} текстов: {time.time()-t0:.1f} сек")
    return np.array(vecs)

doc_texts_full = all_docs["context"].tolist()
doc_emb_full = embed_voyage_full(doc_texts_full, input_type="document")

print("Размерность:", doc_emb_full.shape)

In [ ]:
from pymongo import MongoClient
from pymongo.operations import SearchIndexModel

client = MongoClient(MONGODB_URI)
db = client["rag_project"]
coll_full = db["t2_ragbench_full"]
coll_full.delete_many({})

records = []
for i, row in all_docs.iterrows():
    records.append({
        "context_id": row["context_id"],
        "context": row["context"],
        "company_name": row.get("company_name", ""),
        "embedding_voyage": doc_emb_full[i].tolist(),
    })

# вставка батчами, чтобы не упереться в лимит размера запроса
batch_size = 500
for i in range(0, len(records), batch_size):
    coll_full.insert_many(records[i:i+batch_size])
    print(f"  Записано {min(i+batch_size, len(records))}/{len(records)}")

print(f"Итого записано: {coll_full.count_documents({})} документов")

vector_index_model = SearchIndexModel(
    definition={
        "fields": [
            {"type": "vector", "path": "embedding_voyage", "numDimensions": 1024, "similarity": "cosine"},
        ]
    },
    name="vector_index_full",
    type="vectorSearch",
)
text_index_model = SearchIndexModel(
    definition={"mappings": {"dynamic": False, "fields": {"context": {"type": "string"}}}},
    name="text_index_full",
    type="search",
)

t0 = time.time()
coll_full.create_search_index(vector_index_model)
coll_full.create_search_index(text_index_model)

def wait_index_ready(coll, name, timeout=300):
    start = time.time()
    while time.time() - start < timeout:
        idx = list(coll.list_search_indexes(name))
        if idx and idx[0].get("queryable"):
            return True
        time.sleep(5)
    return False

ready_v = wait_index_ready(coll_full, "vector_index_full")
ready_t = wait_index_ready(coll_full, "text_index_full")
print(f"Индексы готовы за {time.time()-t0:.1f} сек (vector={ready_v}, text={ready_t})")

In [ ]:
db["t2_ragbench_test"].drop()
print("Тестовая коллекция удалена")

# теперь создаём text-индекс для полного корпуса
coll_full.create_search_index(text_index_model)

ready_t = wait_index_ready(coll_full, "text_index_full")
print(f"text_index_full готов: {ready_t}")

In [ ]:
# посмотреть, что реально осталось по индексам в базе
for coll_name in db.list_collection_names():
    c = db[coll_name]
    idxs = list(c.list_search_indexes())
    print(coll_name, "->", [ix.get("name") for ix in idxs])

In [ ]:
coll_full.create_search_index(text_index_model)

ready_t = wait_index_ready(coll_full, "text_index_full")
print(f"text_index_full готов: {ready_t}")

In [ ]:
def wait_index_ready(coll, name, timeout=300):
    start = time.time()
    while time.time() - start < timeout:
        idx = list(coll.list_search_indexes(name))
        if idx and idx[0].get("queryable"):
            return True
        time.sleep(5)
    return False

ready_t = wait_index_ready(coll_full, "text_index_full")
print(f"text_index_full готов: {ready_t}")

In [ ]:
def sample_eval(df, n, seed=42):
    return df.sample(n=min(n, len(df)), random_state=seed)

eval_fq = sample_eval(finqa, 90)
eval_cf = sample_eval(convfinqa, 37)
eval_td = sample_eval(tatdqa, 123)

eval_set = pd.concat([eval_fq, eval_cf, eval_td], ignore_index=True)
print(f"Eval-подвыборка: {len(eval_set)} вопросов")

eval_set.to_parquet(f"{out_dir}/eval_subset_250.parquet")
print("Сохранено в Drive")

In [ ]:
cols = ["id", "context_id", "question", "program_answer", "original_answer"]

eval_fq = sample_eval(finqa, 90)[cols]
eval_cf = sample_eval(convfinqa, 37)[cols]
eval_td = sample_eval(tatdqa, 123)[cols]

eval_set = pd.concat([eval_fq, eval_cf, eval_td], ignore_index=True)
print(f"Eval-подвыборка: {len(eval_set)} вопросов")

eval_set.to_parquet(f"{out_dir}/eval_subset_250.parquet")
print("Сохранено в Drive")

In [ ]:
import math

eval_query_texts = eval_set["question"].tolist()
eval_query_emb = embed_voyage_full(eval_query_texts, input_type="query", batch_size=64)

def hybrid_search(query_text, query_emb, k=5):
    pipeline = [
        {
            "$rankFusion": {
                "input": {
                    "pipelines": {
                        "vectorPipeline": [
                            {
                                "$vectorSearch": {
                                    "index": "vector_index_full",
                                    "path": "embedding_voyage",
                                    "queryVector": query_emb.tolist(),
                                    "numCandidates": 50,
                                    "limit": 10,
                                }
                            }
                        ],
                        "fullTextPipeline": [
                            {"$search": {"index": "text_index_full", "text": {"query": query_text, "path": "context"}}},
                            {"$limit": 10},
                        ],
                    }
                },
                "combination": {"weights": {"vectorPipeline": 0.5, "fullTextPipeline": 0.5}},
            }
        },
        {"$limit": k},
        {"$project": {"context_id": 1, "_id": 0}},
    ]
    return [r["context_id"] for r in coll_full.aggregate(pipeline)]

hits = 0
ndcg_sum = 0
t0 = time.time()
for i, row in eval_set.iterrows():
    retrieved = hybrid_search(row["question"], eval_query_emb[i], k=5)
    if row["context_id"] in retrieved:
        hits += 1
        rank = retrieved.index(row["context_id"]) + 1
        ndcg_sum += 1 / math.log2(rank + 1)
    if (i + 1) % 50 == 0:
        print(f"  {i+1}/{len(eval_set)} обработано, {time.time()-t0:.0f} сек")

recall_at_5 = hits / len(eval_set)
ndcg_at_5 = ndcg_sum / len(eval_set)
print(f"\nBaseline (без reranker, без contextual chunks):")
print(f"Recall@5: {recall_at_5:.3f}")
print(f"NDCG@5:   {ndcg_at_5:.3f}")
print(f"Время: {time.time()-t0:.1f} сек на {len(eval_set)} запросов")

In [ ]:
def get_subset(id_str):
    if id_str.startswith("finqa"):
        return "FinQA"
    elif id_str.startswith("convfinqa"):
        return "ConvFinQA"
    elif id_str.startswith("tatqa"):
        return "TAT-DQA"
    return "unknown"

eval_set["subset"] = eval_set["id"].apply(get_subset)

def hybrid_search_topn(query_text, query_emb, k=10):
    pipeline = [
        {
            "$rankFusion": {
                "input": {
                    "pipelines": {
                        "vectorPipeline": [
                            {"$vectorSearch": {
                                "index": "vector_index_full",
                                "path": "embedding_voyage",
                                "queryVector": query_emb.tolist(),
                                "numCandidates": 50,
                                "limit": k,
                            }}
                        ],
                        "fullTextPipeline": [
                            {"$search": {"index": "text_index_full", "text": {"query": query_text, "path": "context"}}},
                            {"$limit": k},
                        ],
                    }
                },
                "combination": {"weights": {"vectorPipeline": 0.5, "fullTextPipeline": 0.5}},
            }
        },
        {"$limit": k},
        {"$project": {"context_id": 1, "_id": 0}},
    ]
    return [r["context_id"] for r in coll_full.aggregate(pipeline)]

diag = []
t0 = time.time()
for i, row in eval_set.iterrows():
    retrieved = hybrid_search_topn(row["question"], eval_query_emb[i], k=10)
    rank = retrieved.index(row["context_id"]) + 1 if row["context_id"] in retrieved else None
    diag.append({"subset": row["subset"], "rank": rank})
    if (i + 1) % 50 == 0:
        print(f"  {i+1}/{len(eval_set)}, {time.time()-t0:.0f} сек")

diag_df = pd.DataFrame(diag)

print("\nДоля найденных в топ-10 по подсетам:")
print(diag_df.groupby("subset")["rank"].apply(lambda x: x.notna().mean()))

in_top5 = (diag_df["rank"] <= 5).sum()
near_miss = diag_df["rank"].between(6, 10).sum()
total_miss = diag_df["rank"].isna().sum()
print(f"\nВ топ-5: {in_top5}")
print(f"Near-miss (ранг 6-10): {near_miss}")
print(f"Полный промах (не в топ-10): {total_miss}")

In [ ]:
!pip install -q flashrank

from flashrank import Ranker, RerankRequest

ranker = Ranker(model_name="ms-marco-MiniLM-L-12-v2")

context_lookup = dict(zip(all_docs["context_id"], all_docs["context"]))

def hybrid_search_topn_full(query_text, query_emb, k=10):
    pipeline = [
        {
            "$rankFusion": {
                "input": {
                    "pipelines": {
                        "vectorPipeline": [
                            {"$vectorSearch": {
                                "index": "vector_index_full",
                                "path": "embedding_voyage",
                                "queryVector": query_emb.tolist(),
                                "numCandidates": 50,
                                "limit": k,
                            }}
                        ],
                        "fullTextPipeline": [
                            {"$search": {"index": "text_index_full", "text": {"query": query_text, "path": "context"}}},
                            {"$limit": k},
                        ],
                    }
                },
                "combination": {"weights": {"vectorPipeline": 0.5, "fullTextPipeline": 0.5}},
            }
        },
        {"$limit": k},
        {"$project": {"context_id": 1, "_id": 0}},
    ]
    return [r["context_id"] for r in coll_full.aggregate(pipeline)]

reranked_results = []
t0 = time.time()
for i, row in eval_set.iterrows():
    candidates = hybrid_search_topn_full(row["question"], eval_query_emb[i], k=10)
    passages = [{"id": cid, "text": context_lookup.get(cid, "")[:2000]} for cid in candidates]
    rerank_req = RerankRequest(query=row["question"], passages=passages)
    reranked = ranker.rerank(rerank_req)
    reranked_ids = [r["id"] for r in reranked]
    rank_after = reranked_ids.index(row["context_id"]) + 1 if row["context_id"] in reranked_ids else None
    reranked_results.append({"context_id": row["context_id"], "subset": row["subset"], "rank_before": None, "rank_after": rank_after})
    if (i + 1) % 50 == 0:
        print(f"  {i+1}/{len(eval_set)}, {time.time()-t0:.0f} сек")

rerank_df = pd.DataFrame(reranked_results)
rerank_df["rank_before"] = diag_df["rank"].values

hits_after = (rerank_df["rank_after"] <= 5).sum()
ndcg_after = rerank_df["rank_after"].dropna().apply(lambda r: 1/math.log2(r+1) if r <= 5 else 0).sum() / len(rerank_df)
print(f"\nС reranker: Recall@5 = {hits_after/len(rerank_df):.3f}, NDCG@5 = {ndcg_after:.3f}")
print(f"Было (baseline): Recall@5 = 0.808, NDCG@5 = 0.615")

In [ ]:
rerank_df["delta"] = rerank_df["rank_before"].fillna(11) - rerank_df["rank_after"].fillna(11)

improved = (rerank_df["delta"] > 0).sum()
worsened = (rerank_df["delta"] < 0).sum()
same = (rerank_df["delta"] == 0).sum()
print(f"Стало лучше: {improved}, стало хуже: {worsened}, без изменений: {same}")

print("\nПо подсетам — среднее изменение ранга (положительное = лучше):")
print(rerank_df.groupby("subset")["delta"].mean())

# сколько из тех, что были ТОЧНО в топ-5, выпали из топ-5
was_top5 = rerank_df["rank_before"] <= 5
now_not_top5 = (rerank_df["rank_after"] > 5) | rerank_df["rank_after"].isna()
demoted_from_top5 = (was_top5 & now_not_top5).sum()
print(f"\nБыли в топ-5, но reranker выкинул за топ-5: {demoted_from_top5} из {was_top5.sum()}")

In [ ]:
!pip install -q anthropic

In [ ]:
import anthropic

ANTHROPIC_API_KEY = "YOU_KEY_HERE"
claude = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

print("Клиент создан:", claude is not None)


In [ ]:
def sample_for_enrich(df, n, seed=42):
    return df.drop_duplicates("context_id").sample(n=min(n, df["context_id"].nunique()), random_state=seed)

enrich_fq = sample_for_enrich(finqa, 72)
enrich_cf = sample_for_enrich(convfinqa, 30)
enrich_td = sample_for_enrich(tatdqa, 98)

enrich_docs = pd.concat([enrich_fq, enrich_cf, enrich_td], ignore_index=True).drop_duplicates("context_id")
print(f"Документов для обогащения: {len(enrich_docs)}")

In [ ]:
CONTEXT_PROMPT = """Ты помогаешь улучшить поиск по фрагментам финансовых отчётов.
Вот фрагмент документа:

<chunk>
{chunk}
</chunk>

Дай короткую (1-2 предложения) справку: какая компания, какой год отчёта, о чём фрагмент (какие показатели/таблица).
Только справка, без вступлений."""

enriched_contexts = []
input_tokens_total = 0
output_tokens_total = 0
t0 = time.time()

for i, row in enrich_docs.iterrows():
    chunk_text = row["context"][:3000]
    prompt = CONTEXT_PROMPT.format(chunk=chunk_text)

    resp = claude.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=150,
        messages=[{"role": "user", "content": prompt}],
    )
    blurb = resp.content[0].text.strip()
    enriched_contexts.append({
        "context_id": row["context_id"],
        "blurb": blurb,
        "enriched_context": f"{blurb}\n\n{row['context']}",
    })
    input_tokens_total += resp.usage.input_tokens
    output_tokens_total += resp.usage.output_tokens

    if len(enriched_contexts) % 25 == 0:
        print(f"  {len(enriched_contexts)}/{len(enrich_docs)}, {time.time()-t0:.0f} сек")

print(f"\nГотово за {time.time()-t0:.1f} сек")
print(f"Входных токенов: {input_tokens_total}, выходных: {output_tokens_total}")

cost = (input_tokens_total / 1_000_000) * 1.0 + (output_tokens_total / 1_000_000) * 5.0
print(f"Стоимость теста: ~${cost:.4f}")
print(f"Экстраполяция на весь корпус (x{7318/len(enrich_docs):.1f}): ~${cost * 7318/len(enrich_docs):.2f}")

In [ ]:
enrich_ctx_ids = set(enrich_docs["context_id"])
enrich_eval_queries = all_qa[all_qa["context_id"].isin(enrich_ctx_ids)].reset_index(drop=True)
print(f"Вопросов к этим 200 документам: {len(enrich_eval_queries)}")

original_texts = enrich_docs["context"].tolist()
enriched_texts = [e["enriched_context"] for e in enriched_contexts]

orig_doc_emb = embed_voyage_full(original_texts, input_type="document")
enriched_doc_emb = embed_voyage_full(enriched_texts, input_type="document")

query_texts_enrich = enrich_eval_queries["question"].tolist()
query_emb_enrich = embed_voyage_full(query_texts_enrich, input_type="query")

print("Готово:", orig_doc_emb.shape, enriched_doc_emb.shape, query_emb_enrich.shape)

In [ ]:
def recall_at_k_local(doc_emb, query_emb, doc_ctx_ids, query_ctx_ids, k=5):
    doc_emb_norm = doc_emb / np.linalg.norm(doc_emb, axis=1, keepdims=True)
    query_emb_norm = query_emb / np.linalg.norm(query_emb, axis=1, keepdims=True)
    sims = query_emb_norm @ doc_emb_norm.T
    hits = 0
    for i in range(len(query_ctx_ids)):
        top_k_idx = np.argsort(-sims[i])[:k]
        retrieved = {doc_ctx_ids[j] for j in top_k_idx}
        if query_ctx_ids[i] in retrieved:
            hits += 1
    return hits / len(query_ctx_ids)

doc_ctx_ids = enrich_docs["context_id"].tolist()
query_ctx_ids = enrich_eval_queries["context_id"].tolist()

recall_original = recall_at_k_local(orig_doc_emb, query_emb_enrich, doc_ctx_ids, query_ctx_ids)
recall_enriched = recall_at_k_local(enriched_doc_emb, query_emb_enrich, doc_ctx_ids, query_ctx_ids)

print(f"Recall@5 без обогащения: {recall_original:.3f}")
print(f"Recall@5 с обогащением:  {recall_enriched:.3f}")

In [ ]:
def is_numeric(val):
    try:
        float(val)
        return True
    except (ValueError, TypeError):
        return False

numeric_eval = eval_set[eval_set["program_answer"].apply(is_numeric)].copy()
print(f"Вопросов с числовым ответом в eval_set: {len(numeric_eval)}")

test_sample = numeric_eval.sample(n=min(30, len(numeric_eval)), random_state=42).reset_index(drop=True)
print(f"Взято для теста: {len(test_sample)}")
print(test_sample[["question", "program_answer"]].head(5))

In [ ]:
context_lookup = dict(zip(all_docs["context_id"], all_docs["context"]))

POT_PROMPT = """You are given a financial document and a question. Write a short Python script that computes the answer to the question using the numbers found in the document. Assign the final numeric result to a variable called `answer`. Only output the Python code in a code block (```python ... ```), no explanation before or after.

Document:
{context}

Question: {question}"""

DIRECT_PROMPT = """You are given a financial document and a question. Answer the question with brief reasoning, then give the final numeric answer on the last line in the exact format "Final answer: <number>".

Document:
{context}

Question: {question}"""

import re

def extract_pot_code(text):
    match = re.search(r"```(?:python)?\s*(.*?)```", text, re.DOTALL)
    return match.group(1) if match else text

def run_pot_code(code):
    safe_globals = {"__builtins__": {"abs": abs, "round": round, "min": min, "max": max, "sum": sum, "len": len}}
    local_vars = {}
    try:
        exec(code, safe_globals, local_vars)
        return local_vars.get("answer")
    except Exception:
        return None

def extract_direct_answer(text):
    match = re.search(r"Final answer:\s*(-?[\d,]*\.?\d+)", text)
    if match:
        try:
            return float(match.group(1).replace(",", ""))
        except ValueError:
            return None
    return None

def is_close(a, b, tol=0.01):
    if a is None or b is None:
        return False
    try:
        a, b = float(a), float(b)
    except (ValueError, TypeError):
        return False
    if b == 0:
        return abs(a - b) < tol
    return abs(a - b) / abs(b) < tol

results = []
t0 = time.time()

for i, row in test_sample.iterrows():
    context = context_lookup.get(row["context_id"], "")
    gold = row["program_answer"]

    pot_resp = claude.messages.create(
        model="claude-sonnet-5",
        max_tokens=500,
        messages=[{"role": "user", "content": POT_PROMPT.format(context=context, question=row["question"])}],
    )
    pot_code = extract_pot_code(pot_resp.content[0].text)
    pot_answer = run_pot_code(pot_code)
    pot_correct = is_close(pot_answer, gold)

    direct_resp = claude.messages.create(
        model="claude-sonnet-5",
        max_tokens=500,
        messages=[{"role": "user", "content": DIRECT_PROMPT.format(context=context, question=row["question"])}],
    )
    direct_answer = extract_direct_answer(direct_resp.content[0].text)
    direct_correct = is_close(direct_answer, gold)

    results.append({
        "question": row["question"], "gold": gold,
        "pot_answer": pot_answer, "pot_correct": pot_correct,
        "direct_answer": direct_answer, "direct_correct": direct_correct,
    })
    if (i + 1) % 10 == 0:
        print(f"  {i+1}/{len(test_sample)}, {time.time()-t0:.0f} сек")

results_df = pd.DataFrame(results)
print(f"\nГотово за {time.time()-t0:.1f} сек")
print(f"PoT точность:    {results_df['pot_correct'].mean():.3f}")
print(f"Direct точность: {results_df['direct_correct'].mean():.3f}")

In [ ]:
def get_text(resp):
    for block in resp.content:
        if block.type == "text":
            return block.text
    return ""

In [ ]:
results = []
t0 = time.time()

for i, row in test_sample.iterrows():
    context = context_lookup.get(row["context_id"], "")
    gold = row["program_answer"]

    pot_resp = claude.messages.create(
        model="claude-sonnet-5",
        max_tokens=500,
        messages=[{"role": "user", "content": POT_PROMPT.format(context=context, question=row["question"])}],
    )
    pot_code = extract_pot_code(get_text(pot_resp))
    pot_answer = run_pot_code(pot_code)
    pot_correct = is_close(pot_answer, gold)

    direct_resp = claude.messages.create(
        model="claude-sonnet-5",
        max_tokens=500,
        messages=[{"role": "user", "content": DIRECT_PROMPT.format(context=context, question=row["question"])}],
    )
    direct_answer = extract_direct_answer(get_text(direct_resp))
    direct_correct = is_close(direct_answer, gold)

    results.append({
        "question": row["question"], "gold": gold,
        "pot_answer": pot_answer, "pot_correct": pot_correct,
        "direct_answer": direct_answer, "direct_correct": direct_correct,
    })
    if (i + 1) % 10 == 0:
        print(f"  {i+1}/{len(test_sample)}, {time.time()-t0:.0f} сек")

results_df = pd.DataFrame(results)
print(f"\nГотово за {time.time()-t0:.1f} сек")
print(f"PoT точность:    {results_df['pot_correct'].mean():.3f}")
print(f"Direct точность: {results_df['direct_correct'].mean():.3f}")

In [ ]:
def run_pot_code(code):
    safe_builtins = {
        "abs": abs, "round": round, "min": min, "max": max, "sum": sum, "len": len,
        "float": float, "int": int, "str": str, "print": print,
        "list": list, "dict": dict, "sorted": sorted, "range": range,
    }
    safe_globals = {"__builtins__": safe_builtins}
    local_vars = {}
    try:
        exec(code, safe_globals, local_vars)
    except Exception:
        pass  # даже при ошибке на поздней строке answer могла успеть посчитаться
    return local_vars.get("answer")

results = []
raw_outputs = []
t0 = time.time()

for i, row in test_sample.iterrows():
    context = context_lookup.get(row["context_id"], "")
    gold = row["program_answer"]

    pot_resp = claude.messages.create(
        model="claude-sonnet-5", max_tokens=500,
        messages=[{"role": "user", "content": POT_PROMPT.format(context=context, question=row["question"])}],
    )
    pot_text = get_text(pot_resp)
    pot_code = extract_pot_code(pot_text)
    pot_answer = run_pot_code(pot_code)
    pot_correct = is_close(pot_answer, gold)

    direct_resp = claude.messages.create(
        model="claude-sonnet-5", max_tokens=500,
        messages=[{"role": "user", "content": DIRECT_PROMPT.format(context=context, question=row["question"])}],
    )
    direct_text = get_text(direct_resp)
    direct_answer = extract_direct_answer(direct_text)
    direct_correct = is_close(direct_answer, gold)

    results.append({
        "question": row["question"], "gold": gold,
        "pot_answer": pot_answer, "pot_correct": pot_correct,
        "direct_answer": direct_answer, "direct_correct": direct_correct,
    })
    raw_outputs.append({"pot_text": pot_text, "direct_text": direct_text})
    if (i + 1) % 10 == 0:
        print(f"  {i+1}/{len(test_sample)}, {time.time()-t0:.0f} сек")

results_df = pd.DataFrame(results)
print(f"\nГотово за {time.time()-t0:.1f} сек")
print(f"PoT точность:    {results_df['pot_correct'].mean():.3f}")
print(f"Direct точность: {results_df['direct_correct'].mean():.3f}")

In [ ]:
both_correct = ((results_df["pot_correct"]) & (results_df["direct_correct"])).sum()
both_wrong = ((~results_df["pot_correct"]) & (~results_df["direct_correct"])).sum()
pot_only = ((results_df["pot_correct"]) & (~results_df["direct_correct"])).sum()
direct_only = ((~results_df["pot_correct"]) & (results_df["direct_correct"])).sum()

print(f"Оба правильно: {both_correct}")
print(f"Оба неправильно: {both_wrong}")
print(f"Только PoT правильно: {pot_only}")
print(f"Только Direct правильно: {direct_only}")

print("\n--- Примеры, где ОБА ошиблись (посмотреть глазами) ---")
both_wrong_idx = results_df[(~results_df["pot_correct"]) & (~results_df["direct_correct"])].index[:3]
for idx in both_wrong_idx:
    print(f"\nВопрос: {results_df.loc[idx, 'question']}")
    print(f"Правильный ответ: {results_df.loc[idx, 'gold']}")
    print(f"PoT ответ: {results_df.loc[idx, 'pot_answer']}")
    print(f"Direct ответ: {results_df.loc[idx, 'direct_answer']}")
    print(f"--- PoT текст ---\n{raw_outputs[idx]['pot_text'][:500]}")
    print(f"--- Direct текст ---\n{raw_outputs[idx]['direct_text'][:300]}")

In [ ]:
def is_close_v2(a, b, tol=0.01):
    if a is None or b is None:
        return False
    try:
        a, b = float(a), float(b)
    except (ValueError, TypeError):
        return False
    candidates = [b, -b, b*100, -b*100, b/100, -b/100]
    for cb in candidates:
        if cb == 0:
            if abs(a - cb) < tol:
                return True
        elif abs(a - cb) / abs(cb) < tol:
            return True
    return False

results_df["pot_correct_v2"] = results_df.apply(lambda r: is_close_v2(r["pot_answer"], r["gold"]), axis=1)
results_df["direct_correct_v2"] = results_df.apply(lambda r: is_close_v2(r["direct_answer"], r["gold"]), axis=1)

print(f"PoT точность (с поправкой на знак/масштаб):    {results_df['pot_correct_v2'].mean():.3f}")
print(f"Direct точность (с поправкой на знак/масштаб): {results_df['direct_correct_v2'].mean():.3f}")

In [ ]:
JUDGE_PROMPT = """You are evaluating whether a generated answer to a financial question is correct, given the ground truth answer.

Question: {question}
Generated answer: {generated}
Ground truth answer: {gold}

Consider the answer correct if it matches the ground truth value, allowing for minor rounding, sign-convention differences (e.g. -60 vs 60 if direction is ambiguous), or equivalent expression as percentage vs fraction (e.g. 1.5 vs 0.015). Respond with exactly one word: CORRECT or INCORRECT."""

judge_results = []
t0 = time.time()
for i, row in results_df.iterrows():
    prompt = JUDGE_PROMPT.format(question=row["question"], generated=row["direct_answer"], gold=row["gold"])
    resp = claude.messages.create(
        model="claude-sonnet-5", max_tokens=10,
        messages=[{"role": "user", "content": prompt}],
    )
    verdict = get_text(resp).strip().upper()
    judge_correct = "CORRECT" in verdict and "INCORRECT" not in verdict
    judge_results.append(judge_correct)
    if (i+1) % 10 == 0:
        print(f"  {i+1}/{len(results_df)}, {time.time()-t0:.0f} сек")

results_df["judge_correct"] = judge_results
print(f"\nГотово за {time.time()-t0:.1f} сек")
print(f"Claude-judge точность: {results_df['judge_correct'].mean():.3f}")
print(f"Детерминированная (is_close_v2) точность: {results_df['direct_correct_v2'].mean():.3f}")

agreement = (results_df["judge_correct"] == results_df["direct_correct_v2"]).mean()
print(f"Согласие judge vs детерминированный скоринг: {agreement:.3f}")

disagree = results_df[results_df["judge_correct"] != results_df["direct_correct_v2"]]
print(f"\nРасхождения ({len(disagree)}):")
for idx, row in disagree.iterrows():
    print(f"  Q: {row['question'][:80]}")
    print(f"  Gold: {row['gold']}, Direct: {row['direct_answer']}, judge_correct={row['judge_correct']}, tolerance_correct={row['direct_correct_v2']}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from google.colab import userdata

MONGODB_URI = userdata.get('YOU_KEY_HERE')
VOYAGE_API_KEY = userdata.get('YOU_KEY_HERE')
COHERE_API_KEY = userdata.get('YOU_KEY_HERE')  # dashboard.cohere.com -> API keys -> Trial key

In [ ]:
from google.colab import userdata

MONGODB_URI = userdata.get('MONGODB_URI')
VOYAGE_API_KEY = "YOU_KEY_HERE"
COHERE_API_KEY = "YOU_KEY_HERE"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q pymongo voyageai cohere sentence-transformers pandas pyarrow

In [ ]:
import pandas as pd

out_dir = "/content/drive/MyDrive/RAG-project/data/t2-ragbench"
eval_set = pd.read_parquet(f"{out_dir}/eval_subset_250.parquet")
print(f"Eval-вопросов: {len(eval_set)}")

In [ ]:
from pymongo import MongoClient
import voyageai
import time

client = MongoClient(MONGODB_URI)
coll = client["rag_project"]["t2_ragbench_full"]
vo = voyageai.Client(api_key=VOYAGE_API_KEY)

def hybrid_top10(question):
    q_emb = vo.embed([question], model="voyage-4", input_type="query").embeddings[0]
    pipeline = [
        {"$rankFusion": {
            "input": {"pipelines": {
                "vectorPipeline": [
                    {"$vectorSearch": {"index": "vector_index_voyage", "path": "embedding_voyage",
                                        "queryVector": q_emb, "numCandidates": 50, "limit": 10}}
                ],
                "fullTextPipeline": [
                    {"$search": {"index": "text_index", "text": {"query": question, "path": "context"}}},
                    {"$limit": 10},
                ],
            }},
            "combination": {"weights": {"vectorPipeline": 0.5, "fullTextPipeline": 0.5}},
        }},
        {"$limit": 10},
        {"$project": {"context_id": 1, "context": 1, "company_name": 1, "_id": 0}},
    ]
    return list(coll.aggregate(pipeline))

t0 = time.time()
candidates = {}
for i, row in eval_set.iterrows():
    candidates[row["question"]] = hybrid_top10(row["question"])
    if (i + 1) % 50 == 0:
        print(f"  {i+1}/{len(eval_set)}, {time.time()-t0:.0f} сек")
print(f"Готово за {time.time()-t0:.0f} сек")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from google.colab import userdata

MONGODB_URI = userdata.get('MONGODB_URI')
VOYAGE_API_KEY = "YOU_KEY_HERE"
COHERE_API_KEY = "YOU_KEY_HERE"

In [ ]:
!pip install -q pymongo voyageai cohere sentence-transformers pandas pyarrow

In [ ]:
import pandas as pd

out_dir = "/content/drive/MyDrive/RAG-project/data/t2-ragbench"
eval_set = pd.read_parquet(f"{out_dir}/eval_subset_250.parquet")
print(f"Eval-вопросов: {len(eval_set)}")

In [ ]:
!nvidia-smi

In [ ]:
from pymongo import MongoClient
import voyageai
import time

client = MongoClient(MONGODB_URI)
coll = client["rag_project"]["t2_ragbench_full"]
vo = voyageai.Client(api_key=VOYAGE_API_KEY)

def hybrid_top10(question):
    q_emb = vo.embed([question], model="voyage-4", input_type="query").embeddings[0]
    pipeline = [
        {"$rankFusion": {
            "input": {"pipelines": {
                "vectorPipeline": [
                    {"$vectorSearch": {"index": "vector_index_voyage", "path": "embedding_voyage",
                                        "queryVector": q_emb, "numCandidates": 50, "limit": 10}}
                ],
                "fullTextPipeline": [
                    {"$search": {"index": "text_index", "text": {"query": question, "path": "context"}}},
                    {"$limit": 10},
                ],
            }},
            "combination": {"weights": {"vectorPipeline": 0.5, "fullTextPipeline": 0.5}},
        }},
        {"$limit": 10},
        {"$project": {"context_id": 1, "context": 1, "company_name": 1, "_id": 0}},
    ]
    return list(coll.aggregate(pipeline))

t0 = time.time()
candidates = {}
for i, row in eval_set.iterrows():
    candidates[row["question"]] = hybrid_top10(row["question"])
    if (i + 1) % 50 == 0:
        print(f"  {i+1}/{len(eval_set)}, {time.time()-t0:.0f} сек")
print(f"Готово за {time.time()-t0:.0f} сек")


In [ ]:
import numpy as np

def paired_bootstrap(hits_a, hits_b, B=10000, seed=42):
    """hits_a, hits_b: списки 0/1 одинаковой длины (одни и те же вопросы).
    Возвращает (mean_diff, 95% CI, p_value для H0: diff=0)."""
    rng = np.random.default_rng(seed)
    a, b = np.array(hits_a), np.array(hits_b)
    n = len(a)
    diffs = []
    for _ in range(B):
        idx = rng.integers(0, n, n)
        diffs.append(b[idx].mean() - a[idx].mean())
    diffs = np.array(diffs)
    mean_diff = b.mean() - a.mean()
    ci_low, ci_high = np.percentile(diffs, [2.5, 97.5])
    p_value = 2 * min((diffs <= 0).mean(), (diffs >= 0).mean())
    return mean_diff, (ci_low, ci_high), p_value

def recall_at_5(hits_dict, eval_set):
    return [1 if row["context_id"] in {c["context_id"] for c in hits_dict[row["question"]][:5]} else 0
            for _, row in eval_set.iterrows()]

# baseline (без reranker) -- top-5 из hybrid как есть
hits_baseline = recall_at_5(candidates, eval_set)
print(f"Baseline (без reranker) Recall@5: {np.mean(hits_baseline):.3f}")

In [ ]:
sample_q = eval_set.iloc[0]["question"]
print("Ожидаемый context_id:", repr(eval_set.iloc[0]["context_id"]), type(eval_set.iloc[0]["context_id"]))
print("Количество кандидатов для этого вопроса:", len(candidates[sample_q]))
print("Первый кандидат целиком:", candidates[sample_q][0] if candidates[sample_q] else "СПИСОК ПУСТ")

In [ ]:
# Проверка 1: сама коллекция на месте и не пустая?
print("Документов в коллекции:", coll.count_documents({}))

# Проверка 2: какие search-индексы реально существуют и в каком они статусе?
for idx in coll.list_search_indexes():
    print(idx.get("name"), "| type:", idx.get("type"), "| queryable:", idx.get("queryable"), "| status:", idx.get("status"))

# Проверка 3: работает ли vectorSearch отдельно, без rankFusion?
test_emb = vo.embed(["test"], model="voyage-4", input_type="query").embeddings[0]
vs_result = list(coll.aggregate([
    {"$vectorSearch": {"index": "vector_index_voyage", "path": "embedding_voyage",
                        "queryVector": test_emb, "numCandidates": 50, "limit": 5}},
    {"$project": {"context_id": 1, "_id": 0}},
]))
print("vectorSearch отдельно вернул:", len(vs_result), "результатов")

# Проверка 4: работает ли text search отдельно?
ts_result = list(coll.aggregate([
    {"$search": {"index": "text_index", "text": {"query": "revenue", "path": "context"}}},
    {"$limit": 5},
    {"$project": {"context_id": 1, "_id": 0}},
]))
print("text search отдельно вернул:", len(ts_result), "результатов")

In [ ]:
sample_doc = coll.find_one({}, {"_id": 0})
print(list(sample_doc.keys()))

In [ ]:
from pymongo import MongoClient
import voyageai
import time

client = MongoClient(MONGODB_URI)
coll = client["rag_project"]["t2_ragbench_full"]
vo = voyageai.Client(api_key=VOYAGE_API_KEY)

def hybrid_top10(question):
    q_emb = vo.embed([question], model="voyage-4", input_type="query").embeddings[0]
    pipeline = [
        {"$rankFusion": {
            "input": {"pipelines": {
                "vectorPipeline": [
                    {"$vectorSearch": {"index": "vector_index_full", "path": "embedding_voyage",
                                        "queryVector": q_emb, "numCandidates": 50, "limit": 10}}
                ],
                "fullTextPipeline": [
                    {"$search": {"index": "text_index_full", "text": {"query": question, "path": "context"}}},
                    {"$limit": 10},
                ],
            }},
            "combination": {"weights": {"vectorPipeline": 0.5, "fullTextPipeline": 0.5}},
        }},
        {"$limit": 10},
        {"$project": {"context_id": 1, "context": 1, "company_name": 1, "_id": 0}},
    ]
    return list(coll.aggregate(pipeline))

t0 = time.time()
candidates = {}
for i, row in eval_set.iterrows():
    candidates[row["question"]] = hybrid_top10(row["question"])
    if (i + 1) % 50 == 0:
        print(f"  {i+1}/{len(eval_set)}, {time.time()-t0:.0f} сек")
print(f"Готово за {time.time()-t0:.0f} сек")

In [ ]:
import numpy as np

def paired_bootstrap(hits_a, hits_b, B=10000, seed=42):
    """hits_a, hits_b: списки 0/1 одинаковой длины (одни и те же вопросы).
    Возвращает (mean_diff, 95% CI, p_value для H0: diff=0)."""
    rng = np.random.default_rng(seed)
    a, b = np.array(hits_a), np.array(hits_b)
    n = len(a)
    diffs = []
    for _ in range(B):
        idx = rng.integers(0, n, n)
        diffs.append(b[idx].mean() - a[idx].mean())
    diffs = np.array(diffs)
    mean_diff = b.mean() - a.mean()
    ci_low, ci_high = np.percentile(diffs, [2.5, 97.5])
    p_value = 2 * min((diffs <= 0).mean(), (diffs >= 0).mean())
    return mean_diff, (ci_low, ci_high), p_value

def recall_at_5(hits_dict, eval_set):
    return [1 if row["context_id"] in {c["context_id"] for c in hits_dict[row["question"]][:5]} else 0
            for _, row in eval_set.iterrows()]

# baseline (без reranker) -- top-5 из hybrid как есть
hits_baseline = recall_at_5(candidates, eval_set)
print(f"Baseline (без reranker) Recall@5: {np.mean(hits_baseline):.3f}")

In [ ]:
from sentence_transformers import CrossEncoder

bge_reranker = CrossEncoder("BAAI/bge-reranker-v2-m3", device="cuda", max_length=1024)

def rerank_bge(question, docs, top_n=5):
    pairs = [[question, d["context"][:2000]] for d in docs]
    scores = bge_reranker.predict(pairs)
    ranked = sorted(zip(docs, scores), key=lambda x: -x[1])
    return [d for d, s in ranked[:top_n]]

reranked_bge = {}
t0 = time.time()
for i, row in eval_set.iterrows():
    reranked_bge[row["question"]] = rerank_bge(row["question"], candidates[row["question"]])
    if (i + 1) % 50 == 0:
        print(f"  bge: {i+1}/{len(eval_set)}, {time.time()-t0:.0f} сек")

hits_bge = [1 if row["context_id"] in {d["context_id"] for d in reranked_bge[row["question"]]} else 0
            for _, row in eval_set.iterrows()]
print(f"bge-reranker-v2-m3 Recall@5: {np.mean(hits_bge):.3f}")

diff, ci, p = paired_bootstrap(hits_baseline, hits_bge)
print(f"Разница vs baseline: {diff:+.3f}, 95% CI {ci}, p={p:.4f}")

In [ ]:
import cohere

co = cohere.Client(COHERE_API_KEY)

def rerank_cohere(question, docs, top_n=5):
    texts = [d["context"][:2000] for d in docs]
    resp = co.rerank(model="rerank-v4.0-pro", query=question, documents=texts, top_n=top_n)
    return [docs[r.index] for r in resp.results]

reranked_cohere = {}
t0 = time.time()
for i, row in eval_set.iterrows():
    reranked_cohere[row["question"]] = rerank_cohere(row["question"], candidates[row["question"]])
    if (i + 1) % 50 == 0:
        print(f"  cohere: {i+1}/{len(eval_set)}, {time.time()-t0:.0f} сек")
    time.sleep(0.15)  # trial-ключ: 10 req/min лимит -- на всякий случай троттлим

hits_cohere = [1 if row["context_id"] in {d["context_id"] for d in reranked_cohere[row["question"]]} else 0
               for _, row in eval_set.iterrows()]
print(f"Cohere Rerank v4.0 Pro Recall@5: {np.mean(hits_cohere):.3f}")

diff, ci, p = paired_bootstrap(hits_baseline, hits_cohere)
print(f"Разница vs baseline: {diff:+.3f}, 95% CI {ci}, p={p:.4f}")

In [ ]:
import time

for i, row in eval_set.iterrows():
    if row["question"] in reranked_cohere:
        continue  # уже посчитано, пропускаем
    for attempt in range(3):
        try:
            reranked_cohere[row["question"]] = rerank_cohere(row["question"], candidates[row["question"]])
            break
        except Exception as e:
            print(f"  retry {attempt+1} на вопросе {i}: {e}")
            time.sleep(15)
    time.sleep(6.5)  # 10 запросов/мин = 1 раз в 6 сек, берём с запасом
    if (i + 1) % 25 == 0:
        print(f"  cohere: {i+1}/{len(eval_set)}")

hits_cohere = [1 if row["context_id"] in {d["context_id"] for d in reranked_cohere[row["question"]]} else 0
               for _, row in eval_set.iterrows()]
print(f"Cohere Rerank v4.0 Pro Recall@5: {np.mean(hits_cohere):.3f}")

diff, ci, p = paired_bootstrap(hits_baseline, hits_cohere)
print(f"Разница vs baseline: {diff:+.3f}, 95% CI {ci}, p={p:.4f}")

In [ ]:
def regression_summary(hits_before, hits_after, label):
    better = sum(1 for b, a in zip(hits_before, hits_after) if a > b)
    worse = sum(1 for b, a in zip(hits_before, hits_after) if a < b)
    same = len(hits_before) - better - worse
    print(f"{label}: стало лучше={better}, стало хуже={worse}, без изменений={same}")

regression_summary(hits_baseline, hits_bge, "bge-reranker-v2-m3")
regression_summary(hits_baseline, hits_cohere, "Cohere Rerank v4.0 Pro")

# --- Ячейка 9: итог -- решение по развилке 2, теперь с двумя кандидатами и значимостью ---
print("\n=== ИТОГ ===")
print(f"Baseline (hybrid, без reranker):  Recall@5 = {np.mean(hits_baseline):.3f}")
print(f"+ bge-reranker-v2-m3 (бесплатно): Recall@5 = {np.mean(hits_bge):.3f}")
print(f"+ Cohere Rerank v4.0 Pro (платно): Recall@5 = {np.mean(hits_cohere):.3f}")
print("Решение: использовать вариант с положительным и значимым (p<0.05) приростом.")
print("Если оба варианта не дают значимого прироста -- предыдущее решение (без reranker) подтверждается,")
print("но уже на сильных моделях, а не на заведомо слабой FlashRank -- вывод становится надёжнее.")

In [ ]:
!pip install -q anthropic rank_bm25

In [ ]:
def rerank_bge_full(question, docs):
    pairs = [[question, d["context"][:2000]] for d in docs]
    scores = bge_reranker.predict(pairs)
    ranked = sorted(zip(docs, scores), key=lambda x: -x[1])
    return [d for d, s in ranked]  # весь порядок, не обрезаем до top-5

def get_rank(doc_list, context_id):
    for idx, d in enumerate(doc_list):
        if d["context_id"] == context_id:
            return idx + 1
    return None

results = []
for i, row in eval_set.iterrows():
    q = row["question"]
    full_ranked = rerank_bge_full(q, candidates[q])
    results.append({
        "baseline_rank": get_rank(candidates[q], row["context_id"]),
        "bge_rank": get_rank(full_ranked, row["context_id"]),
    })
    if (i + 1) % 50 == 0:
        print(f"{i+1}/{len(eval_set)}")

df_ranks = pd.DataFrame(results)
near_miss = df_ranks[(df_ranks["baseline_rank"] > 5) & (df_ranks["baseline_rank"] <= 10)]
fixed = near_miss[near_miss["bge_rank"] <= 5]
print(f"Near-miss (ранг 6-10) исходно: {len(near_miss)}, из них исправлено: {len(fixed)}")

good = df_ranks[df_ranks["baseline_rank"] <= 5]
broken = good[(good["bge_rank"] > 5) | (good["bge_rank"].isna())]
print(f"Изначально верно (ранг ≤5): {len(good)}, из них сломано: {len(broken)}")

In [ ]:
print("Разбивка сломанных (было ≤5, стало >5) по исходному рангу:")
print(broken["baseline_rank"].value_counts().sort_index())

print("\nРазбивка исправленных (было 6-10, стало ≤5) по исходному рангу:")
print(fixed["baseline_rank"].value_counts().sort_index())

In [ ]:
print("На какое место реально попали исправленные (было 6-10, стало ≤5):")
print(fixed["bge_rank"].value_counts().sort_index())

In [ ]:
lengths = all_q.drop_duplicates("context_id")["context"].str.len()
print(f"Медиана длины context: {lengths.median():.0f} символов")
print(f"Среднее: {lengths.mean():.0f}, 75-й перцентиль: {lengths.quantile(0.75):.0f}, максимум: {lengths.max():.0f}")
print(f"Доля документов длиннее 2000 символов: {(lengths > 2000).mean()*100:.1f}%")

In [ ]:
import pandas as pd

lengths = []
for _, row in eval_set.iterrows():
    for d in candidates[row["question"]]:
        if d["context_id"] == row["context_id"]:
            lengths.append(len(d["context"]))
            break

lengths = pd.Series(lengths)
print(f"Найдено длин: {len(lengths)} из {len(eval_set)}")
print(f"Медиана: {lengths.median():.0f}, среднее: {lengths.mean():.0f}, 75-й перцентиль: {lengths.quantile(0.75):.0f}, максимум: {lengths.max():.0f}")
print(f"Доля документов длиннее 2000 символов: {(lengths > 2000).mean()*100:.1f}%")

In [ ]:
def rerank_cohere_full(question, docs, top_n=5):
    texts = [d["context"] for d in docs]  # без обрезки [:2000]
    resp = co.rerank(model="rerank-v4.0-pro", query=question, documents=texts, top_n=top_n)
    return [docs[r.index] for r in resp.results]

import time

reranked_cohere_full = {}
for i, row in eval_set.iterrows():
    if row["question"] in reranked_cohere_full:
        continue
    for attempt in range(3):
        try:
            reranked_cohere_full[row["question"]] = rerank_cohere_full(row["question"], candidates[row["question"]])
            break
        except Exception as e:
            print(f"  retry {attempt+1} на вопросе {i}: {e}")
            time.sleep(15)
    time.sleep(6.5)
    if (i + 1) % 25 == 0:
        print(f"  cohere-full: {i+1}/{len(eval_set)}")

hits_cohere_full = [1 if row["context_id"] in {d["context_id"] for d in reranked_cohere_full[row["question"]]} else 0
                     for _, row in eval_set.iterrows()]
print(f"Cohere Rerank v4.0 Pro (без обрезки) Recall@5: {np.mean(hits_cohere_full):.3f}")

diff, ci, p = paired_bootstrap(hits_baseline, hits_cohere_full)
print(f"Разница vs baseline: {diff:+.3f}, 95% CI {ci}, p={p:.4f}")

In [ ]:
# --- Найти вопросы с полным промахом (верного документа нет в топ-10 baseline) ---
missed_questions = []
for _, row in eval_set.iterrows():
    doc_ids_in_top10 = {d["context_id"] for d in candidates[row["question"]]}
    if row["context_id"] not in doc_ids_in_top10:
        missed_questions.append(row)

print(f"Вопросов с полным промахом (не в топ-10): {len(missed_questions)}")

# --- Для каждого — расширенный запрос с пулом 100, найти реальный ранг (без reranker, только retrieval) ---
def hybrid_top_n(question, n=100):
    q_emb = vo.embed([question], model="voyage-4", input_type="query").embeddings[0]
    pipeline = [
        {"$rankFusion": {
            "input": {"pipelines": {
                "vectorPipeline": [
                    {"$vectorSearch": {"index": "vector_index_full", "path": "embedding_voyage",
                                        "queryVector": q_emb, "numCandidates": max(500, n*5), "limit": n}}
                ],
                "fullTextPipeline": [
                    {"$search": {"index": "text_index_full", "text": {"query": question, "path": "context"}}},
                    {"$limit": n},
                ],
            }},
            "combination": {"weights": {"vectorPipeline": 0.5, "fullTextPipeline": 0.5}},
        }},
        {"$limit": n},
        {"$project": {"context_id": 1, "_id": 0}},
    ]
    return list(coll.aggregate(pipeline))

import time
ranks_of_missed = []
t0 = time.time()
for row in missed_questions:
    results = hybrid_top_n(row["question"], n=100)
    rank = None
    for idx, d in enumerate(results):
        if d["context_id"] == row["context_id"]:
            rank = idx + 1
            break
    ranks_of_missed.append(rank)  # None = не найден даже в топ-100

print(f"Готово за {time.time()-t0:.0f} сек")

import pandas as pd
ranks_series = pd.Series(ranks_of_missed)
print(f"\nИз {len(missed_questions)} промахов (не в топ-10 при пуле 10):")
print(f"  11-20 место:  {((ranks_series>=11)&(ranks_series<=20)).sum()}  <- доступны уже при пуле 20-50")
print(f"  21-50 место:  {((ranks_series>=21)&(ranks_series<=50)).sum()}  <- доступны при пуле 50")
print(f"  51-100 место: {((ranks_series>=51)&(ranks_series<=100)).sum()}  <- НЕ доступны при пуле 50")
print(f"  >100 (не найдено): {ranks_series.isna().sum()}  <- НЕ доступны даже при пуле 100")

In [ ]:
def hybrid_top50(question):
    q_emb = vo.embed([question], model="voyage-4", input_type="query").embeddings[0]
    pipeline = [
        {"$rankFusion": {
            "input": {"pipelines": {
                "vectorPipeline": [
                    {"$vectorSearch": {"index": "vector_index_full", "path": "embedding_voyage",
                                        "queryVector": q_emb, "numCandidates": 500, "limit": 50}}
                ],
                "fullTextPipeline": [
                    {"$search": {"index": "text_index_full", "text": {"query": question, "path": "context"}}},
                    {"$limit": 50},
                ],
            }},
            "combination": {"weights": {"vectorPipeline": 0.5, "fullTextPipeline": 0.5}},
        }},
        {"$limit": 50},
        {"$project": {"context_id": 1, "context": 1, "company_name": 1, "_id": 0}},
    ]
    return list(coll.aggregate(pipeline))

import time
candidates_50 = {}
t0 = time.time()
for i, row in eval_set.iterrows():
    candidates_50[row["question"]] = hybrid_top50(row["question"])
    if (i + 1) % 50 == 0:
        print(f"  {i+1}/{len(eval_set)}, {time.time()-t0:.0f} сек")
print(f"Готово за {time.time()-t0:.0f} сек")

In [ ]:
def rerank_cohere_pool50(question, docs, top_n=5):
    texts = [d["context"] for d in docs]  # без обрезки
    resp = co.rerank(model="rerank-v4.0-pro", query=question, documents=texts, top_n=top_n)
    return [docs[r.index] for r in resp.results]

reranked_cohere_pool50 = {}
t0 = time.time()
for i, row in eval_set.iterrows():
    if row["question"] in reranked_cohere_pool50:
        continue
    for attempt in range(3):
        try:
            reranked_cohere_pool50[row["question"]] = rerank_cohere_pool50(row["question"], candidates_50[row["question"]])
            break
        except Exception as e:
            print(f"  retry {attempt+1} на вопросе {i}: {e}")
            time.sleep(15)
    time.sleep(6.5)
    if (i + 1) % 25 == 0:
        print(f"  cohere-pool50: {i+1}/{len(eval_set)}")

hits_cohere_pool50 = [1 if row["context_id"] in {d["context_id"] for d in reranked_cohere_pool50[row["question"]]} else 0
                       for _, row in eval_set.iterrows()]
print(f"Cohere Rerank v4.0 Pro (пул=50, без обрезки) Recall@5: {np.mean(hits_cohere_pool50):.3f}")

diff, ci, p = paired_bootstrap(hits_baseline, hits_cohere_pool50)
print(f"Разница vs baseline (0.808): {diff:+.3f}, 95% CI {ci}, p={p:.4f}")

diff2, ci2, p2 = paired_bootstrap(hits_cohere_full, hits_cohere_pool50)
print(f"Разница vs пул=10 без обрезки (0.868): {diff2:+.3f}, 95% CI {ci2}, p={p2:.4f}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q anthropic rank_bm25 voyageai pandas pyarrow numpy

In [ ]:
from google.colab import userdata
import anthropic
import voyageai

ANTHROPIC_API_KEY = "YOU_KEY_HERE"
VOYAGE_API_KEY = userdata.get('YOU_KEY_HERE')

claude = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
vo = voyageai.Client(api_key=VOYAGE_API_KEY)

In [ ]:
from google.colab import userdata
import anthropic
import voyageai

ANTHROPIC_API_KEY = "YOU_KEY_HERE"
VOYAGE_API_KEY = "YOU_KEY_HEREi_key=ANTHROPIC_API_KEY)
vo = voyageai.Client(api_key=VOYAGE_API_KEY)

In [ ]:
import pandas as pd

out_dir = "/content/drive/MyDrive/RAG-project/data/t2-ragbench"

def load_subset(name, splits):
    dfs = [pd.read_parquet(f"{out_dir}/{name}_{s}.parquet") for s in splits]
    return pd.concat(dfs, ignore_index=True)

finqa = load_subset("FinQA", ["train", "dev", "test"])
convfinqa = load_subset("ConvFinQA", ["turn_0"])
tatdqa = load_subset("TAT-DQA", ["train", "dev", "test"])

def sample_for_enrich(df, n, seed=42):
    return df.drop_duplicates("context_id").sample(n=min(n, df["context_id"].nunique()), random_state=seed)

enrich_fq = sample_for_enrich(finqa, 162)
enrich_cf = sample_for_enrich(convfinqa, 68)
enrich_td = sample_for_enrich(tatdqa, 220)

enrich_docs = pd.concat([enrich_fq, enrich_cf, enrich_td], ignore_index=True).drop_duplicates("context_id")
print(f"Документов для обогащения: {len(enrich_docs)}")

all_q = pd.concat([finqa, convfinqa, tatdqa], ignore_index=True)
eval_q = all_q[all_q["context_id"].isin(enrich_docs["context_id"])].reset_index(drop=True)
print(f"Вопросов к этим документам: {len(eval_q)} (цель ~1500, минимум для мощности ~800-1600)")

In [ ]:
CONTEXT_PROMPT = """Ты помогаешь улучшить поиск по фрагментам финансовых отчётов.
Вот фрагмент документа:

<chunk>
{chunk}
</chunk>

Дай короткую (1-2 предложения) справку: какая компания, какой год отчёта, о чём фрагмент (какие показатели/таблица).
Только справка, без вступлений."""

import time

enriched = {}
input_tokens_total = 0
output_tokens_total = 0
t0 = time.time()

for i, row in enrich_docs.reset_index(drop=True).iterrows():
    chunk_text = row["context"][:3000]
    prompt = CONTEXT_PROMPT.format(chunk=chunk_text)
    resp = claude.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=150,
        messages=[{"role": "user", "content": prompt}],
    )
    blurb = resp.content[0].text.strip()
    enriched[row["context_id"]] = f"{blurb}\n\n{row['context']}"
    input_tokens_total += resp.usage.input_tokens
    output_tokens_total += resp.usage.output_tokens
    if (i + 1) % 50 == 0:
        print(f"  {i+1}/{len(enrich_docs)}, {time.time()-t0:.0f} сек")

print(f"\nГотово за {time.time()-t0:.1f} сек")
cost = (input_tokens_total / 1_000_000) * 1.0 + (output_tokens_total / 1_000_000) * 5.0
print(f"Стоимость теста: ~${cost:.4f}")
print(f"Экстраполяция на весь корпус (x{7318/len(enrich_docs):.1f}): ~${cost * 7318/len(enrich_docs):.2f}")

In [ ]:
import numpy as np

doc_ids = enrich_docs["context_id"].tolist()
raw_texts = enrich_docs.set_index("context_id")["context"].to_dict()
enriched_texts = enriched  # dict context_id -> enriched text

def embed_voyage(texts, input_type, batch_size=32, model="voyage-4"):
    vecs = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        result = vo.embed(batch, model=model, input_type=input_type)
        vecs.extend(result.embeddings)
    return np.array(vecs)

raw_doc_texts = [raw_texts[cid] for cid in doc_ids]
enriched_doc_texts = [enriched_texts[cid] for cid in doc_ids]
query_texts = eval_q["question"].tolist()

emb_raw_docs = embed_voyage(raw_doc_texts, input_type="document")
emb_enriched_docs = embed_voyage(enriched_doc_texts, input_type="document")
emb_queries = embed_voyage(query_texts, input_type="query")

print(f"raw docs: {emb_raw_docs.shape}, enriched docs: {emb_enriched_docs.shape}, queries: {emb_queries.shape}")

In [ ]:
from rank_bm25 import BM25Okapi

def tokenize(text):
    return text.lower().split()

bm25_raw = BM25Okapi([tokenize(t) for t in raw_doc_texts])
bm25_enriched = BM25Okapi([tokenize(t) for t in enriched_doc_texts])

print("BM25-индексы (raw и enriched) построены")

In [ ]:
def rrf_ranking(bm25_scores, dense_sims, k=60, top_n=10):
    bm25_rank = {idx: r for r, idx in enumerate(np.argsort(-bm25_scores))}
    dense_rank = {idx: r for r, idx in enumerate(np.argsort(-dense_sims))}
    n = len(bm25_scores)
    rrf_scores = np.zeros(n)
    for idx in range(n):
        rrf_scores[idx] = 1.0 / (k + bm25_rank[idx]) + 1.0 / (k + dense_rank[idx])
    top_idx = np.argsort(-rrf_scores)[:top_n]
    return top_idx

def hybrid_recall_at_5(bm25_index, doc_embeddings, query_embeddings, doc_ids, eval_q, label):
    doc_emb_norm = doc_embeddings / np.linalg.norm(doc_embeddings, axis=1, keepdims=True)
    query_emb_norm = query_embeddings / np.linalg.norm(query_embeddings, axis=1, keepdims=True)
    hits = []
    for i, row in enumerate(eval_q.itertuples()):
        q_tokens = tokenize(row.question)
        bm25_scores = bm25_index.get_scores(q_tokens)
        dense_sims = query_emb_norm[i] @ doc_emb_norm.T
        top_idx = rrf_ranking(bm25_scores, dense_sims, top_n=5)
        retrieved = {doc_ids[j] for j in top_idx}
        hits.append(1 if row.context_id in retrieved else 0)
    print(f"{label} Recall@5: {np.mean(hits):.3f}")
    return hits

import time
t0 = time.time()
hits_baseline = hybrid_recall_at_5(bm25_raw, emb_raw_docs, emb_queries, doc_ids, eval_q, "Baseline hybrid (raw)")
hits_contextual = hybrid_recall_at_5(bm25_enriched, emb_enriched_docs, emb_queries, doc_ids, eval_q, "Contextual hybrid")
print(f"Заняло {time.time()-t0:.0f} сек")

In [ ]:
def paired_bootstrap(hits_a, hits_b, B=10000, seed=42):
    rng = np.random.default_rng(seed)
    a, b = np.array(hits_a), np.array(hits_b)
    n = len(a)
    diffs = []
    for _ in range(B):
        idx = rng.integers(0, n, n)
        diffs.append(b[idx].mean() - a[idx].mean())
    diffs = np.array(diffs)
    mean_diff = b.mean() - a.mean()
    ci_low, ci_high = np.percentile(diffs, [2.5, 97.5])
    p_value = 2 * min((diffs <= 0).mean(), (diffs >= 0).mean())
    return mean_diff, (ci_low, ci_high), p_value

diff, ci, p = paired_bootstrap(hits_baseline, hits_contextual)
print(f"Разница (contextual - baseline): {diff:+.4f}")
print(f"95% CI: [{ci[0]:+.4f}, {ci[1]:+.4f}]")
print(f"p-value: {p:.4f}")

In [ ]:
print("\n=== ИТОГ ===")
print(f"n вопросов = {len(eval_q)}, n документов обогащено = {len(enrich_docs)}")
print(f"Baseline hybrid:    Recall@5 = {np.mean(hits_baseline):.3f}")
print(f"Contextual hybrid:  Recall@5 = {np.mean(hits_contextual):.3f}")
print(f"Стоимость обогащения всего корпуса: ~${cost * 7318/len(enrich_docs):.2f}")
if p < 0.05 and diff > 0:
    print("Значимый положительный эффект (p<0.05) -- решение 2.3 меняется на 'использовать contextual chunks'.")
elif p < 0.05 and diff < 0:
    print("Значимый отрицательный эффект -- решение 'не использовать' подтверждается уже надёжно.")
else:
    print("Эффект статистически не значим даже на увеличенной выборке и на правильном (hybrid) пайплайне --")
    print("решение 'не использовать contextual chunks' подтверждается на прочной основе, не только 'в пределах шума'.")

In [ ]:
import numpy as np

np.random.seed(42)
sample_idx = np.random.choice(len(eval_q), size=150, replace=False)
eval_q_add = eval_q.iloc[sample_idx].reset_index(drop=True)
emb_queries_add = emb_queries[sample_idx]

def local_hybrid_top_n(bm25_index, doc_emb_norm, query_emb_norm, doc_ids, doc_texts, question, n=50, k=60):
    q_tokens = tokenize(question)
    bm25_scores = bm25_index.get_scores(q_tokens)
    dense_sims = query_emb_norm @ doc_emb_norm.T
    bm25_rank = {idx: r for r, idx in enumerate(np.argsort(-bm25_scores))}
    dense_rank = {idx: r for r, idx in enumerate(np.argsort(-dense_sims))}
    rrf_scores = np.zeros(len(doc_ids))
    for idx in range(len(doc_ids)):
        rrf_scores[idx] = 1.0 / (k + bm25_rank[idx]) + 1.0 / (k + dense_rank[idx])
    top_idx = np.argsort(-rrf_scores)[:n]
    return [{"context_id": doc_ids[j], "context": doc_texts[j]} for j in top_idx]

emb_raw_docs_norm = emb_raw_docs / np.linalg.norm(emb_raw_docs, axis=1, keepdims=True)
emb_enriched_docs_norm = emb_enriched_docs / np.linalg.norm(emb_enriched_docs, axis=1, keepdims=True)

candidates_raw_50 = {}
candidates_enriched_50 = {}
for i, row in eval_q_add.iterrows():
    q_emb_norm = emb_queries_add[i] / np.linalg.norm(emb_queries_add[i])
    candidates_raw_50[row["question"]] = local_hybrid_top_n(bm25_raw, emb_raw_docs_norm, q_emb_norm, doc_ids, raw_doc_texts, row["question"], n=50)
    candidates_enriched_50[row["question"]] = local_hybrid_top_n(bm25_enriched, emb_enriched_docs_norm, q_emb_norm, doc_ids, enriched_doc_texts, row["question"], n=50)

print(f"Кандидаты (raw и enriched) готовы для {len(eval_q_add)} вопросов")

In [ ]:
def rerank_cohere(question, docs, top_n=5):
    texts = [d["context"] for d in docs]
    resp = co.rerank(model="rerank-v4.0-pro", query=question, documents=texts, top_n=top_n)
    return [docs[r.index] for r in resp.results]

import time

reranked_raw = {}
reranked_enriched = {}
t0 = time.time()
count = 0
for i, row in eval_q_add.iterrows():
    q = row["question"]
    if q not in reranked_raw:
        for attempt in range(3):
            try:
                reranked_raw[q] = rerank_cohere(q, candidates_raw_50[q])
                break
            except Exception as e:
                print(f"  retry raw {attempt+1} на {i}: {e}")
                time.sleep(15)
        time.sleep(6.5)
    if q not in reranked_enriched:
        for attempt in range(3):
            try:
                reranked_enriched[q] = rerank_cohere(q, candidates_enriched_50[q])
                break
            except Exception as e:
                print(f"  retry enriched {attempt+1} на {i}: {e}")
                time.sleep(15)
        time.sleep(6.5)
    count += 1
    if count % 25 == 0:
        print(f"  {count}/{len(eval_q_add)}, {time.time()-t0:.0f} сек")

print(f"Готово за {time.time()-t0:.0f} сек")

In [ ]:
import cohere
COHERE_API_KEY = "YOU_KEY_HERE"
co = cohere.Client(COHERE_API_KEY)
print("Cohere-клиент создан:", co is not None)

In [ ]:
!pip install -q cohere

import cohere
COHERE_API_KEY = "YOU_KEY_HERE"
co = cohere.Client(COHERE_API_KEY)
print("Cohere-клиент создан:", co is not None)

In [ ]:
!pip install -q cohere

import cohere
COHERE_API_KEY = "YOU_KEY_HERE"
co = cohere.Client(COHERE_API_KEY)
print("Cohere-клиент создан:", co is not None)

In [ ]:
def rerank_cohere(question, docs, top_n=5):
    texts = [d["context"] for d in docs]
    resp = co.rerank(model="rerank-v4.0-pro", query=question, documents=texts, top_n=top_n)
    return [docs[r.index] for r in resp.results]

import time

reranked_raw = {}
reranked_enriched = {}
t0 = time.time()
count = 0
for i, row in eval_q_add.iterrows():
    q = row["question"]
    if q not in reranked_raw:
        for attempt in range(3):
            try:
                reranked_raw[q] = rerank_cohere(q, candidates_raw_50[q])
                break
            except Exception as e:
                print(f"  retry raw {attempt+1} на {i}: {e}")
                time.sleep(15)
        time.sleep(6.5)
    if q not in reranked_enriched:
        for attempt in range(3):
            try:
                reranked_enriched[q] = rerank_cohere(q, candidates_enriched_50[q])
                break
            except Exception as e:
                print(f"  retry enriched {attempt+1} на {i}: {e}")
                time.sleep(15)
        time.sleep(6.5)
    count += 1
    if count % 25 == 0:
        print(f"  {count}/{len(eval_q_add)}, {time.time()-t0:.0f} сек")

print(f"Готово за {time.time()-t0:.0f} сек")

In [ ]:
np.random.seed(42)
sample_idx = np.random.choice(len(eval_q), size=150, replace=False)
eval_q_add = eval_q.iloc[sample_idx].reset_index(drop=True)
emb_queries_add = emb_queries[sample_idx]

candidates_raw_50 = {}
candidates_enriched_50 = {}
for i, row in eval_q_add.iterrows():
    q_emb_norm = emb_queries_add[i] / np.linalg.norm(emb_queries_add[i])
    candidates_raw_50[row["question"]] = local_hybrid_top_n(bm25_raw, emb_raw_docs_norm, q_emb_norm, doc_ids, raw_doc_texts, row["question"], n=50)
    candidates_enriched_50[row["question"]] = local_hybrid_top_n(bm25_enriched, emb_enriched_docs_norm, q_emb_norm, doc_ids, enriched_doc_texts, row["question"], n=50)

print(f"Кандидаты готовы для {len(eval_q_add)} вопросов")

In [ ]:
import numpy as np
for name in ['eval_q', 'enrich_docs', 'enriched', 'emb_raw_docs', 'emb_enriched_docs', 'emb_queries', 'bm25_raw', 'bm25_enriched', 'doc_ids', 'co', 'vo', 'claude']:
    print(name, name in dir())

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q anthropic rank_bm25 voyageai cohere pandas pyarrow numpy

In [ ]:
import anthropic, voyageai, cohere

ANTHROPIC_API_KEY = "YOU_KEY_HERE"
VOYAGE_API_KEY = "YOU_KEY_HERE"
COHERE_API_KEY = "YOU_KEY_HERE"

claude = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
vo = voyageai.Client(api_key=VOYAGE_API_KEY)
co = cohere.Client(api_key=COHERE_API_KEY)
print("Клиенты созданы")

In [ ]:
import pandas as pd

out_dir = "/content/drive/MyDrive/RAG-project/data/t2-ragbench"

def load_subset(name, splits):
    dfs = [pd.read_parquet(f"{out_dir}/{name}_{s}.parquet") for s in splits]
    return pd.concat(dfs, ignore_index=True)

finqa = load_subset("FinQA", ["train", "dev", "test"])
convfinqa = load_subset("ConvFinQA", ["turn_0"])
tatdqa = load_subset("TAT-DQA", ["train", "dev", "test"])

def sample_for_enrich(df, n, seed=42):
    return df.drop_duplicates("context_id").sample(n=min(n, df["context_id"].nunique()), random_state=seed)

enrich_fq = sample_for_enrich(finqa, 162)
enrich_cf = sample_for_enrich(convfinqa, 68)
enrich_td = sample_for_enrich(tatdqa, 220)
enrich_docs = pd.concat([enrich_fq, enrich_cf, enrich_td], ignore_index=True).drop_duplicates("context_id")

all_q = pd.concat([finqa, convfinqa, tatdqa], ignore_index=True)
eval_q = all_q[all_q["context_id"].isin(enrich_docs["context_id"])].reset_index(drop=True)
print(f"Документов: {len(enrich_docs)}, вопросов: {len(eval_q)}")

In [ ]:
CONTEXT_PROMPT = """Ты помогаешь улучшить поиск по фрагментам финансовых отчётов.
Вот фрагмент документа:

<chunk>
{chunk}
</chunk>

Дай короткую (1-2 предложения) справку: какая компания, какой год отчёта, о чём фрагмент (какие показатели/таблица).
Только справка, без вступлений."""

import time, json

enriched = {}
t0 = time.time()
for i, row in enrich_docs.reset_index(drop=True).iterrows():
    chunk_text = row["context"][:3000]
    prompt = CONTEXT_PROMPT.format(chunk=chunk_text)
    resp = claude.messages.create(model="claude-haiku-4-5-20251001", max_tokens=150, messages=[{"role": "user", "content": prompt}])
    blurb = resp.content[0].text.strip()
    enriched[row["context_id"]] = f"{blurb}\n\n{row['context']}"
    if (i + 1) % 50 == 0:
        print(f"  {i+1}/{len(enrich_docs)}, {time.time()-t0:.0f} сек")

print(f"Готово за {time.time()-t0:.1f} сек")

with open("/content/drive/MyDrive/RAG-project/data/t2-ragbench/enriched_450.json", "w") as f:
    json.dump(enriched, f)
print("Сохранено на Drive")

In [ ]:
import json
with open("/content/drive/MyDrive/RAG-project/data/t2-ragbench/enriched_450.json", "w") as f:
    json.dump(enriched, f)
print(f"Сохранено частично: {len(enriched)} записей")

In [ ]:
import time

t0 = time.time()
count = 0
for i, row in enrich_docs.reset_index(drop=True).iterrows():
    if row["context_id"] in enriched:
        continue
    chunk_text = row["context"][:3000]
    prompt = CONTEXT_PROMPT.format(chunk=chunk_text)
    resp = claude.messages.create(model="claude-haiku-4-5-20251001", max_tokens=150, messages=[{"role": "user", "content": prompt}])
    blurb = resp.content[0].text.strip()
    enriched[row["context_id"]] = f"{blurb}\n\n{row['context']}"
    count += 1
    if count % 50 == 0:
        with open("/content/drive/MyDrive/RAG-project/data/t2-ragbench/enriched_450.json", "w") as f:
            json.dump(enriched, f)
        print(f"  {len(enriched)}/450, {time.time()-t0:.0f} сек (сохранено)")

with open("/content/drive/MyDrive/RAG-project/data/t2-ragbench/enriched_450.json", "w") as f:
    json.dump(enriched, f)
print(f"Готово: {len(enriched)}/450 за {time.time()-t0:.1f} сек, сохранено")

In [ ]:
import numpy as np

doc_ids = enrich_docs["context_id"].tolist()
raw_texts = enrich_docs.set_index("context_id")["context"].to_dict()
enriched_texts = enriched

def embed_voyage(texts, input_type, batch_size=32, model="voyage-4"):
    vecs = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        result = vo.embed(batch, model=model, input_type=input_type)
        vecs.extend(result.embeddings)
    return np.array(vecs)

raw_doc_texts = [raw_texts[cid] for cid in doc_ids]
enriched_doc_texts = [enriched_texts[cid] for cid in doc_ids]
query_texts = eval_q["question"].tolist()

emb_raw_docs = embed_voyage(raw_doc_texts, input_type="document")
emb_enriched_docs = embed_voyage(enriched_doc_texts, input_type="document")
emb_queries = embed_voyage(query_texts, input_type="query")

print(f"raw docs: {emb_raw_docs.shape}, enriched docs: {emb_enriched_docs.shape}, queries: {emb_queries.shape}")

In [ ]:
from rank_bm25 import BM25Okapi

def tokenize(text):
    return text.lower().split()

bm25_raw = BM25Okapi([tokenize(t) for t in raw_doc_texts])
bm25_enriched = BM25Okapi([tokenize(t) for t in enriched_doc_texts])

print("BM25-индексы построены")

In [ ]:
np.random.seed(42)
sample_idx = np.random.choice(len(eval_q), size=150, replace=False)
eval_q_add = eval_q.iloc[sample_idx].reset_index(drop=True)
emb_queries_add = emb_queries[sample_idx]

def local_hybrid_top_n(bm25_index, doc_emb_norm, query_emb_norm, doc_ids, doc_texts, question, n=50, k=60):
    q_tokens = tokenize(question)
    bm25_scores = bm25_index.get_scores(q_tokens)
    dense_sims = query_emb_norm @ doc_emb_norm.T
    bm25_rank = {idx: r for r, idx in enumerate(np.argsort(-bm25_scores))}
    dense_rank = {idx: r for r, idx in enumerate(np.argsort(-dense_sims))}
    rrf_scores = np.zeros(len(doc_ids))
    for idx in range(len(doc_ids)):
        rrf_scores[idx] = 1.0 / (k + bm25_rank[idx]) + 1.0 / (k + dense_rank[idx])
    top_idx = np.argsort(-rrf_scores)[:n]
    return [{"context_id": doc_ids[j], "context": doc_texts[j]} for j in top_idx]

emb_raw_docs_norm = emb_raw_docs / np.linalg.norm(emb_raw_docs, axis=1, keepdims=True)
emb_enriched_docs_norm = emb_enriched_docs / np.linalg.norm(emb_enriched_docs, axis=1, keepdims=True)

candidates_raw_50 = {}
candidates_enriched_50 = {}
for i, row in eval_q_add.iterrows():
    q_emb_norm = emb_queries_add[i] / np.linalg.norm(emb_queries_add[i])
    candidates_raw_50[row["question"]] = local_hybrid_top_n(bm25_raw, emb_raw_docs_norm, q_emb_norm, doc_ids, raw_doc_texts, row["question"], n=50)
    candidates_enriched_50[row["question"]] = local_hybrid_top_n(bm25_enriched, emb_enriched_docs_norm, q_emb_norm, doc_ids, enriched_doc_texts, row["question"], n=50)

print(f"Кандидаты готовы для {len(eval_q_add)} вопросов")

In [ ]:
import time, json

save_path_raw = "/content/drive/MyDrive/RAG-project/data/t2-ragbench/reranked_raw_150.json"
save_path_enriched = "/content/drive/MyDrive/RAG-project/data/t2-ragbench/reranked_enriched_150.json"

def rerank_cohere(question, docs, top_n=5):
    texts = [d["context"] for d in docs]
    resp = co.rerank(model="rerank-v4.0-pro", query=question, documents=texts, top_n=top_n)
    return [docs[r.index] for r in resp.results]

reranked_raw = {}
reranked_enriched = {}
t0 = time.time()
count = 0
for i, row in eval_q_add.iterrows():
    q = row["question"]
    if q not in reranked_raw:
        for attempt in range(3):
            try:
                reranked_raw[q] = rerank_cohere(q, candidates_raw_50[q])
                break
            except Exception as e:
                print(f"  retry raw {attempt+1} на {i}: {e}")
                time.sleep(15)
        time.sleep(6.5)
    if q not in reranked_enriched:
        for attempt in range(3):
            try:
                reranked_enriched[q] = rerank_cohere(q, candidates_enriched_50[q])
                break
            except Exception as e:
                print(f"  retry enriched {attempt+1} на {i}: {e}")
                time.sleep(15)
        time.sleep(6.5)
    count += 1
    if count % 25 == 0:
        with open(save_path_raw, "w") as f:
            json.dump(reranked_raw, f)
        with open(save_path_enriched, "w") as f:
            json.dump(reranked_enriched, f)
        print(f"  {count}/{len(eval_q_add)}, {time.time()-t0:.0f} сек (сохранено)")

with open(save_path_raw, "w") as f:
    json.dump(reranked_raw, f)
with open(save_path_enriched, "w") as f:
    json.dump(reranked_enriched, f)
print(f"Готово за {time.time()-t0:.0f} сек, сохранено")

In [ ]:
import json, time, os

save_path_raw = "/content/drive/MyDrive/RAG-project/data/t2-ragbench/reranked_raw_150.json"
save_path_enriched = "/content/drive/MyDrive/RAG-project/data/t2-ragbench/reranked_enriched_150.json"

# Подхватить уже посчитанное с диска
if os.path.exists(save_path_raw):
    with open(save_path_raw) as f:
        reranked_raw = json.load(f)
else:
    reranked_raw = {}

if os.path.exists(save_path_enriched):
    with open(save_path_enriched) as f:
        reranked_enriched = json.load(f)
else:
    reranked_enriched = {}

print(f"Загружено с диска: raw={len(reranked_raw)}, enriched={len(reranked_enriched)}")

def rerank_cohere(question, docs, top_n=5):
    texts = [d["context"] for d in docs]
    resp = co.rerank(model="rerank-v4.0-pro", query=question, documents=texts, top_n=top_n)
    return [docs[r.index] for r in resp.results]

t0 = time.time()
count = 0
for i, row in eval_q_add.iterrows():
    q = row["question"]
    if q not in reranked_raw:
        for attempt in range(3):
            try:
                reranked_raw[q] = rerank_cohere(q, candidates_raw_50[q])
                break
            except Exception as e:
                print(f"  retry raw {attempt+1} на {i}: {e}")
                time.sleep(15)
        time.sleep(6.5)
    if q not in reranked_enriched:
        for attempt in range(3):
            try:
                reranked_enriched[q] = rerank_cohere(q, candidates_enriched_50[q])
                break
            except Exception as e:
                print(f"  retry enriched {attempt+1} на {i}: {e}")
                time.sleep(15)
        time.sleep(6.5)
    count += 1
    if count % 25 == 0:
        with open(save_path_raw, "w") as f:
            json.dump(reranked_raw, f)
        with open(save_path_enriched, "w") as f:
            json.dump(reranked_enriched, f)
        print(f"  прогресс: raw={len(reranked_raw)}, enriched={len(reranked_enriched)} из {len(eval_q_add)}, {time.time()-t0:.0f} сек (сохранено)")

with open(save_path_raw, "w") as f:
    json.dump(reranked_raw, f)
with open(save_path_enriched, "w") as f:
    json.dump(reranked_enriched, f)
print(f"Готово: raw={len(reranked_raw)}, enriched={len(reranked_enriched)}, за {time.time()-t0:.0f} сек")


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q anthropic rank_bm25 voyageai cohere pandas pyarrow numpy

In [ ]:
import anthropic, voyageai, cohere

ANTHROPIC_API_KEY = "YOU_KEY_HERE"
VOYAGE_API_KEY = "YOU_KEY_HERE"
COHERE_API_KEY = "YOY_KEY_HERE"

claude = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
vo = voyageai.Client(api_key=VOYAGE_API_KEY)
co = cohere.Client(api_key=COHERE_API_KEY)
print("Клиенты созданы")

In [ ]:
import pandas as pd

out_dir = "/content/drive/MyDrive/RAG-project/data/t2-ragbench"

def load_subset(name, splits):
    dfs = [pd.read_parquet(f"{out_dir}/{name}_{s}.parquet") for s in splits]
    return pd.concat(dfs, ignore_index=True)

finqa = load_subset("FinQA", ["train", "dev", "test"])
convfinqa = load_subset("ConvFinQA", ["turn_0"])
tatdqa = load_subset("TAT-DQA", ["train", "dev", "test"])

def sample_for_enrich(df, n, seed=42):
    return df.drop_duplicates("context_id").sample(n=min(n, df["context_id"].nunique()), random_state=seed)

enrich_fq = sample_for_enrich(finqa, 162)
enrich_cf = sample_for_enrich(convfinqa, 68)
enrich_td = sample_for_enrich(tatdqa, 220)
enrich_docs = pd.concat([enrich_fq, enrich_cf, enrich_td], ignore_index=True).drop_duplicates("context_id")

all_q = pd.concat([finqa, convfinqa, tatdqa], ignore_index=True)
eval_q = all_q[all_q["context_id"].isin(enrich_docs["context_id"])].reset_index(drop=True)
print(f"Документов: {len(enrich_docs)}, вопросов: {len(eval_q)}")

In [ ]:
import json
with open("/content/drive/MyDrive/RAG-project/data/t2-ragbench/enriched_450.json") as f:
    enriched = json.load(f)
print(f"Обогащение загружено: {len(enriched)} документов")

In [ ]:
import numpy as np

doc_ids = enrich_docs["context_id"].tolist()
raw_texts = enrich_docs.set_index("context_id")["context"].to_dict()
enriched_texts = enriched

def embed_voyage(texts, input_type, batch_size=32, model="voyage-4"):
    vecs = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        result = vo.embed(batch, model=model, input_type=input_type)
        vecs.extend(result.embeddings)
    return np.array(vecs)

raw_doc_texts = [raw_texts[cid] for cid in doc_ids]
enriched_doc_texts = [enriched_texts[cid] for cid in doc_ids]
query_texts = eval_q["question"].tolist()

emb_raw_docs = embed_voyage(raw_doc_texts, input_type="document")
emb_enriched_docs = embed_voyage(enriched_doc_texts, input_type="document")
emb_queries = embed_voyage(query_texts, input_type="query")

print(f"raw docs: {emb_raw_docs.shape}, enriched docs: {emb_enriched_docs.shape}, queries: {emb_queries.shape}")

In [ ]:
from rank_bm25 import BM25Okapi

def tokenize(text):
    return text.lower().split()

bm25_raw = BM25Okapi([tokenize(t) for t in raw_doc_texts])
bm25_enriched = BM25Okapi([tokenize(t) for t in enriched_doc_texts])

print("BM25-индексы построены")

In [ ]:
np.random.seed(42)
sample_idx = np.random.choice(len(eval_q), size=150, replace=False)
eval_q_add = eval_q.iloc[sample_idx].reset_index(drop=True)
emb_queries_add = emb_queries[sample_idx]

def local_hybrid_top_n(bm25_index, doc_emb_norm, query_emb_norm, doc_ids, doc_texts, question, n=50, k=60):
    q_tokens = tokenize(question)
    bm25_scores = bm25_index.get_scores(q_tokens)
    dense_sims = query_emb_norm @ doc_emb_norm.T
    bm25_rank = {idx: r for r, idx in enumerate(np.argsort(-bm25_scores))}
    dense_rank = {idx: r for r, idx in enumerate(np.argsort(-dense_sims))}
    rrf_scores = np.zeros(len(doc_ids))
    for idx in range(len(doc_ids)):
        rrf_scores[idx] = 1.0 / (k + bm25_rank[idx]) + 1.0 / (k + dense_rank[idx])
    top_idx = np.argsort(-rrf_scores)[:n]
    return [{"context_id": doc_ids[j], "context": doc_texts[j]} for j in top_idx]

emb_raw_docs_norm = emb_raw_docs / np.linalg.norm(emb_raw_docs, axis=1, keepdims=True)
emb_enriched_docs_norm = emb_enriched_docs / np.linalg.norm(emb_enriched_docs, axis=1, keepdims=True)

candidates_raw_50 = {}
candidates_enriched_50 = {}
for i, row in eval_q_add.iterrows():
    q_emb_norm = emb_queries_add[i] / np.linalg.norm(emb_queries_add[i])
    candidates_raw_50[row["question"]] = local_hybrid_top_n(bm25_raw, emb_raw_docs_norm, q_emb_norm, doc_ids, raw_doc_texts, row["question"], n=50)
    candidates_enriched_50[row["question"]] = local_hybrid_top_n(bm25_enriched, emb_enriched_docs_norm, q_emb_norm, doc_ids, enriched_doc_texts, row["question"], n=50)

print(f"Кандидаты готовы для {len(eval_q_add)} вопросов")

In [ ]:
import json, time, os

save_path_raw = "/content/drive/MyDrive/RAG-project/data/t2-ragbench/reranked_raw_150.json"
save_path_enriched = "/content/drive/MyDrive/RAG-project/data/t2-ragbench/reranked_enriched_150.json"

if os.path.exists(save_path_raw):
    with open(save_path_raw) as f:
        reranked_raw = json.load(f)
else:
    reranked_raw = {}

if os.path.exists(save_path_enriched):
    with open(save_path_enriched) as f:
        reranked_enriched = json.load(f)
else:
    reranked_enriched = {}

print(f"Загружено с диска: raw={len(reranked_raw)}, enriched={len(reranked_enriched)}")

def rerank_cohere(question, docs, top_n=5):
    texts = [d["context"] for d in docs]
    resp = co.rerank(model="rerank-v4.0-pro", query=question, documents=texts, top_n=top_n)
    return [docs[r.index] for r in resp.results]

t0 = time.time()
count = 0
for i, row in eval_q_add.iterrows():
    q = row["question"]
    if q not in reranked_raw:
        for attempt in range(3):
            try:
                reranked_raw[q] = rerank_cohere(q, candidates_raw_50[q])
                break
            except Exception as e:
                print(f"  retry raw {attempt+1} на {i}: {e}")
                time.sleep(15)
        time.sleep(6.5)
    if q not in reranked_enriched:
        for attempt in range(3):
            try:
                reranked_enriched[q] = rerank_cohere(q, candidates_enriched_50[q])
                break
            except Exception as e:
                print(f"  retry enriched {attempt+1} на {i}: {e}")
                time.sleep(15)
        time.sleep(6.5)
    count += 1
    if count % 25 == 0:
        with open(save_path_raw, "w") as f:
            json.dump(reranked_raw, f)
        with open(save_path_enriched, "w") as f:
            json.dump(reranked_enriched, f)
        print(f"  прогресс: raw={len(reranked_raw)}, enriched={len(reranked_enriched)} из {len(eval_q_add)}, {time.time()-t0:.0f} сек (сохранено)")

with open(save_path_raw, "w") as f:
    json.dump(reranked_raw, f)
with open(save_path_enriched, "w") as f:
    json.dump(reranked_enriched, f)
print(f"Готово: raw={len(reranked_raw)}, enriched={len(reranked_enriched)}, за

In [ ]:
import json, time, os

save_path_raw = "/content/drive/MyDrive/RAG-project/data/t2-ragbench/reranked_raw_150.json"
save_path_enriched = "/content/drive/MyDrive/RAG-project/data/t2-ragbench/reranked_enriched_150.json"

if os.path.exists(save_path_raw):
    with open(save_path_raw) as f:
        reranked_raw = json.load(f)
else:
    reranked_raw = {}

if os.path.exists(save_path_enriched):
    with open(save_path_enriched) as f:
        reranked_enriched = json.load(f)
else:
    reranked_enriched = {}

print("Загружено с диска: raw=" + str(len(reranked_raw)) + ", enriched=" + str(len(reranked_enriched)))

def rerank_cohere(question, docs, top_n=5):
    texts = [d["context"] for d in docs]
    resp = co.rerank(model="rerank-v4.0-pro", query=question, documents=texts, top_n=top_n)
    return [docs[r.index] for r in resp.results]

t0 = time.time()
count = 0
for i, row in eval_q_add.iterrows():
    q = row["question"]
    if q not in reranked_raw:
        for attempt in range(3):
            try:
                reranked_raw[q] = rerank_cohere(q, candidates_raw_50[q])
                break
            except Exception as e:
                print("  retry raw " + str(attempt + 1) + " на " + str(i) + ": " + str(e))
                time.sleep(15)
        time.sleep(6.5)
    if q not in reranked_enriched:
        for attempt in range(3):
            try:
                reranked_enriched[q] = rerank_cohere(q, candidates_enriched_50[q])
                break
            except Exception as e:
                print("  retry enriched " + str(attempt + 1) + " на " + str(i) + ": " + str(e))
                time.sleep(15)
        time.sleep(6.5)
    count += 1
    if count % 25 == 0:
        with open(save_path_raw, "w") as f:
            json.dump(reranked_raw, f)
        with open(save_path_enriched, "w") as f:
            json.dump(reranked_enriched, f)
        elapsed = int(time.time() - t0)
        print("  прогресс: raw=" + str(len(reranked_raw)) + ", enriched=" + str(len(reranked_enriched)) + " из " + str(len(eval_q_add)) + ", " + str(elapsed) + " сек (сохранено)")

with open(save_path_raw, "w") as f:
    json.dump(reranked_raw, f)
with open(save_path_enriched, "w") as f:
    json.dump(reranked_enriched, f)
elapsed = int(time.time() - t0)
print("Готово: raw=" + str(len(reranked_raw)) + ", enriched=" + str(len(reranked_enriched)) + ", за " + str(elapsed) + " сек")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q anthropic rank_bm25 voyageai cohere pandas pyarrow numpy

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q anthropic rank_bm25 voyageai cohere pandas pyarrow numpy

In [ ]:
import anthropic, voyageai, cohere

ANTHROPIC_API_KEY = "YOU_KEY_HERE"
VOYAGE_API_KEY = "YOU_KEY_HERE"
COHERE_API_KEY = "YOU_KEY_HERE"

claude = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
vo = voyageai.Client(api_key=VOYAGE_API_KEY)
co = cohere.Client(api_key=COHERE_API_KEY)
print("Клиенты созданы")

In [ ]:
import pandas as pd

out_dir = "/content/drive/MyDrive/RAG-project/data/t2-ragbench"

def load_subset(name, splits):
    dfs = [pd.read_parquet(f"{out_dir}/{name}_{s}.parquet") for s in splits]
    return pd.concat(dfs, ignore_index=True)

finqa = load_subset("FinQA", ["train", "dev", "test"])
convfinqa = load_subset("ConvFinQA", ["turn_0"])
tatdqa = load_subset("TAT-DQA", ["train", "dev", "test"])

def sample_for_enrich(df, n, seed=42):
    return df.drop_duplicates("context_id").sample(n=min(n, df["context_id"].nunique()), random_state=seed)

enrich_fq = sample_for_enrich(finqa, 162)
enrich_cf = sample_for_enrich(convfinqa, 68)
enrich_td = sample_for_enrich(tatdqa, 220)
enrich_docs = pd.concat([enrich_fq, enrich_cf, enrich_td], ignore_index=True).drop_duplicates("context_id")

all_q = pd.concat([finqa, convfinqa, tatdqa], ignore_index=True)
eval_q = all_q[all_q["context_id"].isin(enrich_docs["context_id"])].reset_index(drop=True)
print(f"Документов: {len(enrich_docs)}, вопросов: {len(eval_q)}")

In [ ]:
import json
with open("/content/drive/MyDrive/RAG-project/data/t2-ragbench/enriched_450.json") as f:
    enriched = json.load(f)
print(f"Обогащение загружено: {len(enriched)} документов")

In [ ]:
import numpy as np

doc_ids = enrich_docs["context_id"].tolist()
raw_texts = enrich_docs.set_index("context_id")["context"].to_dict()
enriched_texts = enriched

def embed_voyage(texts, input_type, batch_size=32, model="voyage-4"):
    vecs = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        result = vo.embed(batch, model=model, input_type=input_type)
        vecs.extend(result.embeddings)
    return np.array(vecs)

raw_doc_texts = [raw_texts[cid] for cid in doc_ids]
enriched_doc_texts = [enriched_texts[cid] for cid in doc_ids]
query_texts = eval_q["question"].tolist()

emb_raw_docs = embed_voyage(raw_doc_texts, input_type="document")
emb_enriched_docs = embed_voyage(enriched_doc_texts, input_type="document")
emb_queries = embed_voyage(query_texts, input_type="query")

print(f"raw docs: {emb_raw_docs.shape}, enriched docs: {emb_enriched_docs.shape}, queries: {emb_queries.shape}")

In [ ]:
from rank_bm25 import BM25Okapi

def tokenize(text):
    return text.lower().split()

bm25_raw = BM25Okapi([tokenize(t) for t in raw_doc_texts])
bm25_enriched = BM25Okapi([tokenize(t) for t in enriched_doc_texts])

print("BM25-индексы построены")

In [ ]:
np.random.seed(42)
sample_idx = np.random.choice(len(eval_q), size=150, replace=False)
eval_q_add = eval_q.iloc[sample_idx].reset_index(drop=True)
emb_queries_add = emb_queries[sample_idx]

def local_hybrid_top_n(bm25_index, doc_emb_norm, query_emb_norm, doc_ids, doc_texts, question, n=50, k=60):
    q_tokens = tokenize(question)
    bm25_scores = bm25_index.get_scores(q_tokens)
    dense_sims = query_emb_norm @ doc_emb_norm.T
    bm25_rank = {idx: r for r, idx in enumerate(np.argsort(-bm25_scores))}
    dense_rank = {idx: r for r, idx in enumerate(np.argsort(-dense_sims))}
    rrf_scores = np.zeros(len(doc_ids))
    for idx in range(len(doc_ids)):
        rrf_scores[idx] = 1.0 / (k + bm25_rank[idx]) + 1.0 / (k + dense_rank[idx])
    top_idx = np.argsort(-rrf_scores)[:n]
    return [{"context_id": doc_ids[j], "context": doc_texts[j]} for j in top_idx]

emb_raw_docs_norm = emb_raw_docs / np.linalg.norm(emb_raw_docs, axis=1, keepdims=True)
emb_enriched_docs_norm = emb_enriched_docs / np.linalg.norm(emb_enriched_docs, axis=1, keepdims=True)

candidates_raw_50 = {}
candidates_enriched_50 = {}
for i, row in eval_q_add.iterrows():
    q_emb_norm = emb_queries_add[i] / np.linalg.norm(emb_queries_add[i])
    candidates_raw_50[row["question"]] = local_hybrid_top_n(bm25_raw, emb_raw_docs_norm, q_emb_norm, doc_ids, raw_doc_texts, row["question"], n=50)
    candidates_enriched_50[row["question"]] = local_hybrid_top_n(bm25_enriched, emb_enriched_docs_norm, q_emb_norm, doc_ids, enriched_doc_texts, row["question"], n=50)

print(f"Кандидаты готовы для {len(eval_q_add)} вопросов")

In [ ]:
import json, time, os

save_path_raw = "/content/drive/MyDrive/RAG-project/data/t2-ragbench/reranked_raw_150.json"
save_path_enriched = "/content/drive/MyDrive/RAG-project/data/t2-ragbench/reranked_enriched_150.json"

if os.path.exists(save_path_raw):
    with open(save_path_raw) as f:
        reranked_raw = json.load(f)
else:
    reranked_raw = {}

if os.path.exists(save_path_enriched):
    with open(save_path_enriched) as f:
        reranked_enriched = json.load(f)
else:
    reranked_enriched = {}

print("Загружено с диска: raw=" + str(len(reranked_raw)) + ", enriched=" + str(len(reranked_enriched)))

def rerank_cohere(question, docs, top_n=5):
    texts = [d["context"] for d in docs]
    resp = co.rerank(model="rerank-v4.0-pro", query=question, documents=texts, top_n=top_n)
    return [docs[r.index] for r in resp.results]

t0 = time.time()
count = 0
for i, row in eval_q_add.iterrows():
    q = row["question"]
    if q not in reranked_raw:
        for attempt in range(3):
            try:
                reranked_raw[q] = rerank_cohere(q, candidates_raw_50[q])
                break
            except Exception as e:
                print("  retry raw " + str(attempt + 1) + " на " + str(i) + ": " + str(e))
                time.sleep(15)
        time.sleep(6.5)
    if q not in reranked_enriched:
        for attempt in range(3):
            try:
                reranked_enriched[q] = rerank_cohere(q, candidates_enriched_50[q])
                break
            except Exception as e:
                print("  retry enriched " + str(attempt + 1) + " на " + str(i) + ": " + str(e))
                time.sleep(15)
        time.sleep(6.5)
    count += 1
    if count % 25 == 0:
        with open(save_path_raw, "w") as f:
            json.dump(reranked_raw, f)
        with open(save_path_enriched, "w") as f:
            json.dump(reranked_enriched, f)
        elapsed = int(time.time() - t0)
        print("  прогресс: raw=" + str(len(reranked_raw)) + ", enriched=" + str(len(reranked_enriched)) + " из " + str(len(eval_q_add)) + ", " + str(elapsed) + " сек (сохранено)")

with open(save_path_raw, "w") as f:
    json.dump(reranked_raw, f)
with open(save_path_enriched, "w") as f:
    json.dump(reranked_enriched, f)
elapsed = int(time.time() - t0)
print("Готово: raw=" + str(len(reranked_raw)) + ", enriched=" + str(len(reranked_enriched)) + ", за " + str(elapsed) + " сек")

In [ ]:
import anthropic, voyageai, cohere

ANTHROPIC_API_KEY = "YOU_KEY_HERE"
VOYAGE_API_KEY = "YOU_KEY_HERE"
COHERE_API_KEY = "YOU_KEY_HERE"

claude = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
vo = voyageai.Client(api_key=VOYAGE_API_KEY)
co = cohere.Client(api_key=COHERE_API_KEY)
print("Клиенты созданы")

In [ ]:
import pandas as pd

out_dir = "/content/drive/MyDrive/RAG-project/data/t2-ragbench"

def load_subset(name, splits):
    dfs = [pd.read_parquet(f"{out_dir}/{name}_{s}.parquet") for s in splits]
    return pd.concat(dfs, ignore_index=True)

finqa = load_subset("FinQA", ["train", "dev", "test"])
convfinqa = load_subset("ConvFinQA", ["turn_0"])
tatdqa = load_subset("TAT-DQA", ["train", "dev", "test"])

def sample_for_enrich(df, n, seed=42):
    return df.drop_duplicates("context_id").sample(n=min(n, df["context_id"].nunique()), random_state=seed)

enrich_fq = sample_for_enrich(finqa, 162)
enrich_cf = sample_for_enrich(convfinqa, 68)
enrich_td = sample_for_enrich(tatdqa, 220)
enrich_docs = pd.concat([enrich_fq, enrich_cf, enrich_td], ignore_index=True).drop_duplicates("context_id")

all_q = pd.concat([finqa, convfinqa, tatdqa], ignore_index=True)
eval_q = all_q[all_q["context_id"].isin(enrich_docs["context_id"])].reset_index(drop=True)
print(f"Документов: {len(enrich_docs)}, вопросов: {len(eval_q)}")

In [ ]:
import json
with open("/content/drive/MyDrive/RAG-project/data/t2-ragbench/enriched_450.json") as f:
    enriched = json.load(f)
print(f"Обогащение загружено: {len(enriched)} документов")

In [ ]:
import numpy as np

doc_ids = enrich_docs["context_id"].tolist()
raw_texts = enrich_docs.set_index("context_id")["context"].to_dict()
enriched_texts = enriched

def embed_voyage(texts, input_type, batch_size=32, model="voyage-4"):
    vecs = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        result = vo.embed(batch, model=model, input_type=input_type)
        vecs.extend(result.embeddings)
    return np.array(vecs)

raw_doc_texts = [raw_texts[cid] for cid in doc_ids]
enriched_doc_texts = [enriched_texts[cid] for cid in doc_ids]
query_texts = eval_q["question"].tolist()

emb_raw_docs = embed_voyage(raw_doc_texts, input_type="document")
emb_enriched_docs = embed_voyage(enriched_doc_texts, input_type="document")
emb_queries = embed_voyage(query_texts, input_type="query")

print(f"raw docs: {emb_raw_docs.shape}, enriched docs: {emb_enriched_docs.shape}, queries: {emb_queries.shape}")

In [ ]:
from rank_bm25 import BM25Okapi

def tokenize(text):
    return text.lower().split()

bm25_raw = BM25Okapi([tokenize(t) for t in raw_doc_texts])
bm25_enriched = BM25Okapi([tokenize(t) for t in enriched_doc_texts])

print("BM25-индексы построены")

In [ ]:
np.random.seed(42)
sample_idx = np.random.choice(len(eval_q), size=150, replace=False)
eval_q_add = eval_q.iloc[sample_idx].reset_index(drop=True)
emb_queries_add = emb_queries[sample_idx]

def local_hybrid_top_n(bm25_index, doc_emb_norm, query_emb_norm, doc_ids, doc_texts, question, n=50, k=60):
    q_tokens = tokenize(question)
    bm25_scores = bm25_index.get_scores(q_tokens)
    dense_sims = query_emb_norm @ doc_emb_norm.T
    bm25_rank = {idx: r for r, idx in enumerate(np.argsort(-bm25_scores))}
    dense_rank = {idx: r for r, idx in enumerate(np.argsort(-dense_sims))}
    rrf_scores = np.zeros(len(doc_ids))
    for idx in range(len(doc_ids)):
        rrf_scores[idx] = 1.0 / (k + bm25_rank[idx]) + 1.0 / (k + dense_rank[idx])
    top_idx = np.argsort(-rrf_scores)[:n]
    return [{"context_id": doc_ids[j], "context": doc_texts[j]} for j in top_idx]

emb_raw_docs_norm = emb_raw_docs / np.linalg.norm(emb_raw_docs, axis=1, keepdims=True)
emb_enriched_docs_norm = emb_enriched_docs / np.linalg.norm(emb_enriched_docs, axis=1, keepdims=True)

candidates_raw_50 = {}
candidates_enriched_50 = {}
for i, row in eval_q_add.iterrows():
    q_emb_norm = emb_queries_add[i] / np.linalg.norm(emb_queries_add[i])
    candidates_raw_50[row["question"]] = local_hybrid_top_n(bm25_raw, emb_raw_docs_norm, q_emb_norm, doc_ids, raw_doc_texts, row["question"], n=50)
    candidates_enriched_50[row["question"]] = local_hybrid_top_n(bm25_enriched, emb_enriched_docs_norm, q_emb_norm, doc_ids, enriched_doc_texts, row["question"], n=50)

print(f"Кандидаты готовы для {len(eval_q_add)} вопросов")

In [ ]:
import json, time, os

save_path_raw = "/content/drive/MyDrive/RAG-project/data/t2-ragbench/reranked_raw_150.json"
save_path_enriched = "/content/drive/MyDrive/RAG-project/data/t2-ragbench/reranked_enriched_150.json"

if os.path.exists(save_path_raw):
    with open(save_path_raw) as f:
        reranked_raw = json.load(f)
else:
    reranked_raw = {}

if os.path.exists(save_path_enriched):
    with open(save_path_enriched) as f:
        reranked_enriched = json.load(f)
else:
    reranked_enriched = {}

print("Загружено с диска: raw=" + str(len(reranked_raw)) + ", enriched=" + str(len(reranked_enriched)))

def rerank_cohere(question, docs, top_n=5):
    texts = [d["context"] for d in docs]
    resp = co.rerank(model="rerank-v4.0-pro", query=question, documents=texts, top_n=top_n)
    return [docs[r.index] for r in resp.results]

t0 = time.time()
count = 0
for i, row in eval_q_add.iterrows():
    q = row["question"]
    if q not in reranked_raw:
        for attempt in range(3):
            try:
                reranked_raw[q] = rerank_cohere(q, candidates_raw_50[q])

In [ ]:
import json, time, os

save_path_raw = "/content/drive/MyDrive/RAG-project/data/t2-ragbench/reranked_raw_150.json"
save_path_enriched = "/content/drive/MyDrive/RAG-project/data/t2-ragbench/reranked_enriched_150.json"

if os.path.exists(save_path_raw):
    with open(save_path_raw) as f:
        reranked_raw = json.load(f)
else:
    reranked_raw = {}

if os.path.exists(save_path_enriched):
    with open(save_path_enriched) as f:
        reranked_enriched = json.load(f)
else:
    reranked_enriched = {}

print("Загружено с диска: raw=" + str(len(reranked_raw)) + ", enriched=" + str(len(reranked_enriched)))

def rerank_cohere(question, docs, top_n=5):
    texts = [d["context"] for d in docs]
    resp = co.rerank(model="rerank-v4.0-pro", query=question, documents=texts, top_n=top_n)
    return [docs[r.index] for r in resp.results]

t0 = time.time()
count = 0
for i, row in eval_q_add.iterrows():
    q = row["question"]
    if q not in reranked_raw:
        for attempt in range(3):
            try:
                reranked_raw[q] = rerank_cohere(q, candidates_raw_50[q])
                break
            except Exception as e:
                print("  retry raw " + str(attempt + 1) + " на " + str(i) + ": " + str(e))
                time.sleep(15)
        time.sleep(6.5)
    if q not in reranked_enriched:
        for attempt in range(3):
            try:
                reranked_enriched[q] = rerank_cohere(q, candidates_enriched_50[q])
                break
            except Exception as e:
                print("  retry enriched " + str(attempt + 1) + " на " + str(i) + ": " + str(e))
                time.sleep(15)
        time.sleep(6.5)
    count += 1
    if count % 25 == 0:
        with open(save_path_raw, "w") as f:
            json.dump(reranked_raw, f)
        with open(save_path_enriched, "w") as f:
            json.dump(reranked_enriched, f)
        elapsed = int(time.time() - t0)
        print("  прогресс: raw=" + str(len(reranked_raw)) + ", enriched=" + str(len(reranked_enriched)) + " из " + str(len(eval_q_add)) + ", " + str(elapsed) + " сек (сохранено)")

with open(save_path_raw, "w") as f:
    json.dump(reranked_raw, f)
with open(save_path_enriched, "w") as f:
    json.dump(reranked_enriched, f)
elapsed = int(time.time() - t0)
print("Готово: raw=" + str(len(reranked_raw)) + ", enriched=" + str(len(reranked_enriched)) + ", за " + str(elapsed) + " сек")

In [ ]:
import numpy as np

def recall_at_k(eval_df, results_dict, k=5, id_key="context_id"):
    hits = 0
    flips = {}
    for i, row in eval_df.iterrows():
        q = row["question"]
        gold_id = row[id_key]
        top_ids = [d["context_id"] for d in results_dict[q][:k]]
        hit = gold_id in top_ids
        hits += hit
        flips[q] = hit
    return hits / len(eval_df), flips

baseline_raw_r5, baseline_raw_flips = recall_at_k(eval_q_add, candidates_raw_50, k=5)
baseline_enriched_r5, baseline_enriched_flips = recall_at_k(eval_q_add, candidates_enriched_50, k=5)
reranked_raw_r5, reranked_raw_flips = recall_at_k(eval_q_add, reranked_raw, k=5)
reranked_enriched_r5, reranked_enriched_flips = recall_at_k(eval_q_add, reranked_enriched, k=5)

print(f"Baseline hybrid (raw, до reranker'а)     Recall@5: {baseline_raw_r5:.3f}")
print(f"Cohere Rerank v4.0 Pro (raw, пул=50, без обрезки) Recall@5: {reranked_raw_r5:.3f}")
print(f"Baseline hybrid (enriched, до reranker'а) Recall@5: {baseline_enriched_r5:.3f}")
print(f"Cohere Rerank v4.0 Pro (enriched, пул=50, без обрезки) Recall@5: {reranked_enriched_r5:.3f}")

def paired_bootstrap_p(flips_a, flips_b, questions, n_boot=10000, seed=42):
    rng = np.random.default_rng(seed)
    a = np.array([flips_a[q] for q in questions], dtype=float)
    b = np.array([flips_b[q] for q in questions], dtype=float)
    obs_diff = b.mean() - a.mean()
    n = len(questions)
    count = 0
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        diff = b[idx].mean() - a[idx].mean()
        if obs_diff >= 0 and diff <= 0:
            count += 1
        elif obs_diff < 0 and diff >= 0:
            count += 1
    return 2 * count / n_boot

questions = eval_q_add["question"].tolist()
p_raw = paired_bootstrap_p(baseline_raw_flips, reranked_raw_flips, questions)
p_enriched = paired_bootstrap_p(baseline_enriched_flips, reranked_enriched_flips, questions)
print(f"\nЗначимость (raw, paired bootstrap):     p={p_raw:.4f}")
print(f"Значимость (enriched, paired bootstrap): p={p_enriched:.4f}")

def regression(baseline_flips, reranked_flips, questions):
    fixed = sum(1 for q in questions if not baseline_flips[q] and reranked_flips[q])
    broken = sum(1 for q in questions if baseline_flips[q] and not reranked_flips[q])
    return fixed, broken

fixed_raw, broken_raw = regression(baseline_raw_flips, reranked_raw_flips, questions)
fixed_enr, broken_enr = regression(baseline_enriched_flips, reranked_enriched_flips, questions)
print(f"\nRaw:      исправлено {fixed_raw}, сломано {broken_raw}")
print(f"Enriched: исправлено {fixed_enr}, сломано {broken_enr}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q pymongo voyageai cohere
import pymongo, voyageai, cohere, pandas as pd, numpy as np, json, time, os

MONGODB_URI = "YOU_KEY_HERE"
VOYAGE_API_KEY = "YOU_KEY_HERE"
COHERE_API_KEY = "YOU_KEY_HERE"

client = pymongo.MongoClient(MONGODB_URI)
coll = client["rag_project"]["t2_ragbench_full"]
vo = voyageai.Client(api_key=VOYAGE_API_KEY)
co = cohere.Client(api_key=COHERE_API_KEY)

out_dir = "/content/drive/MyDrive/RAG-project/data/t2-ragbench"

In [ ]:
save_path_eval900 = f"{out_dir}/eval_subset_900.parquet"

if os.path.exists(save_path_eval900):
    eval_set = pd.read_parquet(save_path_eval900)
    print(f"Выборка загружена с диска: {len(eval_set)} вопросов")
else:
    def load_subset(name, splits):
        dfs = [pd.read_parquet(f"{out_dir}/{name}_{s}.parquet") for s in splits]
        return pd.concat(dfs, ignore_index=True)

    finqa = load_subset("FinQA", ["train", "dev", "test"])
    convfinqa = load_subset("ConvFinQA", ["turn_0"])
    tatdqa = load_subset("TAT-DQA", ["train", "dev", "test"])

    def sample_n(df, n, seed=43):
        return df.sample(n=min(n, len(df)), random_state=seed)

    sample_fq = sample_n(finqa, 324)
    sample_cf = sample_n(convfinqa, 133)
    sample_td = sample_n(tatdqa, 443)
    eval_set = pd.concat([sample_fq, sample_cf, sample_td], ignore_index=True)
    eval_set.to_parquet(save_path_eval900)
    print(f"Выборка создана: {len(eval_set)} вопросов (FinQA {len(sample_fq)}, ConvFinQA {len(sample_cf)}, TAT-DQA {len(sample_td)})")

In [ ]:
save_path_eval900 = f"{out_dir}/eval_subset_900.parquet"

if os.path.exists(save_path_eval900):
    eval_set = pd.read_parquet(save_path_eval900)
    print(f"Выборка загружена с диска: {len(eval_set)} вопросов")
else:
    def load_subset(name, splits):
        dfs = [pd.read_parquet(f"{out_dir}/{name}_{s}.parquet") for s in splits]
        return pd.concat(dfs, ignore_index=True)

    finqa = load_subset("FinQA", ["train", "dev", "test"])
    convfinqa = load_subset("ConvFinQA", ["turn_0"])
    tatdqa = load_subset("TAT-DQA", ["train", "dev", "test"])

    def sample_n(df, n, seed=43):
        return df.sample(n=min(n, len(df)), random_state=seed)

    sample_fq = sample_n(finqa, 324)
    sample_cf = sample_n(convfinqa, 133)
    sample_td = sample_n(tatdqa, 443)
    eval_set = pd.concat([sample_fq, sample_cf, sample_td], ignore_index=True)
    eval_set = eval_set[["question", "context_id"]].copy()
    eval_set.to_parquet(save_path_eval900)
    print(f"Выборка создана: {len(eval_set)} вопросов (FinQA {len(sample_fq)}, ConvFinQA {len(sample_cf)}, TAT-DQA {len(sample_td)})")

In [ ]:
def hybrid_top50(question):
    q_emb = vo.embed([question], model="voyage-4", input_type="query").embeddings[0]
    pipeline = [
        {"$rankFusion": {
            "input": {"pipelines": {
                "vectorPipeline": [
                    {"$vectorSearch": {"index": "vector_index_full", "path": "embedding_voyage",
                                        "queryVector": q_emb, "numCandidates": 200, "limit": 50}}
                ],
                "fullTextPipeline": [
                    {"$search": {"index": "text_index_full", "text": {"query": question, "path": "context"}}},
                    {"$limit": 50},
                ],
            }},
            "combination": {"weights": {"vectorPipeline": 0.5, "fullTextPipeline": 0.5}},
        }},
        {"$limit": 50},
        {"$project": {"context_id": 1, "context": 1, "_id": 0}},
    ]
    return list(coll.aggregate(pipeline))

save_path_candidates = f"{out_dir}/candidates_top50_900.json"
if os.path.exists(save_path_candidates):
    with open(save_path_candidates) as f:
        candidates50 = json.load(f)
    print(f"Кандидаты загружены с диска: {len(candidates50)}")
else:
    candidates50 = {}

t0 = time.time()
count = 0
for i, row in eval_set.iterrows():
    q = row["question"]
    if q not in candidates50:
        candidates50[q] = hybrid_top50(q)
    count += 1
    if count % 50 == 0:
        with open(save_path_candidates, "w") as f:
            json.dump(candidates50, f)
        elapsed = int(time.time() - t0)
        print(f"  кандидаты: {count}/{len(eval_set)}, {elapsed} сек (сохранено)")
with open(save_path_candidates, "w") as f:
    json.dump(candidates50, f)
print(f"Кандидаты готовы: {len(candidates50)}, за {int(time.time()-t0)} сек")

In [ ]:
def rerank_cohere(question, docs, top_n=10):
    texts = [d["context"] for d in docs]
    resp = co.rerank(model="rerank-v4.0-pro", query=question, documents=texts, top_n=top_n)
    return [docs[r.index] for r in resp.results]

save_path_reranked50 = f"{out_dir}/reranked_pool50_900.json"
if os.path.exists(save_path_reranked50):
    with open(save_path_reranked50) as f:
        reranked50 = json.load(f)
else:
    reranked50 = {}

t0 = time.time()
count = 0
for i, row in eval_set.iterrows():
    q = row["question"]
    if q not in reranked50:
        reranked50[q] = rerank_cohere(q, candidates50[q], top_n=10)
    count += 1
    if count % 50 == 0:
        with open(save_path_reranked50, "w") as f:
            json.dump(reranked50, f)
        elapsed = int(time.time() - t0)
        print(f"  rerank пул=50: {count}/{len(eval_set)}, {elapsed} сек (сохранено)")
with open(save_path_reranked50, "w") as f:
    json.dump(reranked50, f)
print(f"Готово: {len(reranked50)}, за {int(time.time()-t0)} сек")

In [ ]:
save_path_reranked10 = f"{out_dir}/reranked_pool10_900.json"
if os.path.exists(save_path_reranked10):
    with open(save_path_reranked10) as f:
        reranked10 = json.load(f)
else:
    reranked10 = {}

t0 = time.time()
count = 0
for i, row in eval_set.iterrows():
    q = row["question"]
    if q not in reranked10:
        top10_candidates = candidates50[q][:10]
        reranked10[q] = rerank_cohere(q, top10_candidates, top_n=5)
    count += 1
    if count % 50 == 0:
        with open(save_path_reranked10, "w") as f:
            json.dump(reranked10, f)
        elapsed = int(time.time() - t0)
        print(f"  rerank пул=10: {count}/{len(eval_set)}, {elapsed} сек (сохранено)")
with open(save_path_reranked10, "w") as f:
    json.dump(reranked10, f)
print(f"Готово: {len(reranked10)}, за {int(time.time()-t0)} сек")

In [ ]:
def hit_flags(results_dict, eval_df, k=5, id_field="context_id"):
    flags = {}
    for _, row in eval_df.iterrows():
        q = row["question"]
        top_ids = [d["context_id"] for d in results_dict[q][:k]]
        flags[q] = 1 if row["context_id"] in top_ids else 0
    return flags

def hit_flags_baseline(candidates_dict, eval_df, k=5):
    flags = {}
    for _, row in eval_df.iterrows():
        q = row["question"]
        top_ids = [d["context_id"] for d in candidates_dict[q][:k]]
        flags[q] = 1 if row["context_id"] in top_ids else 0
    return flags

flags_baseline = hit_flags_baseline(candidates50, eval_set, k=5)
flags_pool10 = hit_flags(reranked10, eval_set, k=5)
flags_pool50 = hit_flags(reranked50, eval_set, k=5)

questions = eval_set["question"].tolist()
n = len(questions)

recall_baseline = sum(flags_baseline.values()) / n
recall10 = sum(flags_pool10.values()) / n
recall50 = sum(flags_pool50.values()) / n
print(f"n = {n}")
print(f"Baseline (без reranker) Recall@5: {recall_baseline:.4f}")
print(f"Пул=10 Recall@5: {recall10:.4f}")
print(f"Пул=50 Recall@5: {recall50:.4f}")

def exact_mcnemar(flags_a, flags_b, questions):
    a = b = c = d = 0
    for q in questions:
        fa, fb = flags_a[q], flags_b[q]
        if fa == 1 and fb == 1: a += 1
        elif fa == 1 and fb == 0: b += 1
        elif fa == 0 and fb == 1: c += 1
        else: d += 1
    return a, b, c, d

from scipy.stats import chi2, binomtest

print("\n=== Пул=10 vs Пул=50 ===")
a, b, c, d = exact_mcnemar(flags_pool10, flags_pool50, questions)
print(f"a(оба верно)={a}, b(сломано)={b}, c(спасено)={c}, d(оба неверно)={d}")
print(f"Discordant pairs: b+c={b+c} ({(b+c)/n*100:.1f}%)")
if b + c > 0:
    stat = (abs(b - c) - 1) ** 2 / (b + c)
    p_chi2 = 1 - chi2.cdf(stat, df=1)
    p_exact = binomtest(min(b, c), b + c, 0.5).pvalue
else:
    p_chi2 = p_exact = 1.0
print(f"McNemar chi2 p-value: {p_chi2:.4f}")
print(f"McNemar exact (binomial) p-value: {p_exact:.4f}")

print("\n=== Baseline vs Пул=50 (для полноты, ожидаемо значимо) ===")
a2, b2, c2, d2 = exact_mcnemar(flags_baseline, flags_pool50, questions)
if b2 + c2 > 0:
    stat2 = (abs(b2 - c2) - 1) **

In [ ]:
def hit_flags(results_dict, eval_df, k=5, id_field="context_id"):
    flags = {}
    for _, row in eval_df.iterrows():
        q = row["question"]
        top_ids = [d["context_id"] for d in results_dict[q][:k]]
        flags[q] = 1 if row["context_id"] in top_ids else 0
    return flags

def hit_flags_baseline(candidates_dict, eval_df, k=5):
    flags = {}
    for _, row in eval_df.iterrows():
        q = row["question"]
        top_ids = [d["context_id"] for d in candidates_dict[q][:k]]
        flags[q] = 1 if row["context_id"] in top_ids else 0
    return flags

flags_baseline = hit_flags_baseline(candidates50, eval_set, k=5)
flags_pool10 = hit_flags(reranked10, eval_set, k=5)
flags_pool50 = hit_flags(reranked50, eval_set, k=5)

questions = eval_set["question"].tolist()
n = len(questions)

recall_baseline = sum(flags_baseline.values()) / n
recall10 = sum(flags_pool10.values()) / n
recall50 = sum(flags_pool50.values()) / n
print(f"n = {n}")
print(f"Baseline (без reranker) Recall@5: {recall_baseline:.4f}")
print(f"Пул=10 Recall@5: {recall10:.4f}")
print(f"Пул=50 Recall@5: {recall50:.4f}")

def exact_mcnemar(flags_a, flags_b, questions):
    a = b = c = d = 0
    for q in questions:
        fa, fb = flags_a[q], flags_b[q]
        if fa == 1 and fb == 1: a += 1
        elif fa == 1 and fb == 0: b += 1
        elif fa == 0 and fb == 1: c += 1
        else: d += 1
    return a, b, c, d

from scipy.stats import chi2, binomtest

print("\n=== Пул=10 vs Пул=50 ===")
a, b, c, d = exact_mcnemar(flags_pool10, flags_pool50, questions)
print(f"a(оба верно)={a}, b(сломано)={b}, c(спасено)={c}, d(оба неверно)={d}")
print(f"Discordant pairs: b+c={b+c} ({(b+c)/n*100:.1f}%)")
if b + c > 0:
    diff = abs(b - c) - 1
    stat = (diff * diff) / (b + c)
    p_chi2 = 1 - chi2.cdf(stat, df=1)
    p_exact = binomtest(min(b, c), b + c, 0.5).pvalue
else:
    p_chi2 = p_exact = 1.0
print(f"McNemar chi2 p-value: {p_chi2:.4f}")
print(f"McNemar exact (binomial) p-value: {p_exact:.4f}")

print("\n=== Baseline vs Пул=50 (для полноты, ожидаемо значимо) ===")
a2, b2, c2, d2 = exact_mcnemar(flags_baseline, flags_pool50, questions)
if b2 + c2 > 0:
    diff2 = abs(b2 - c2) - 1
    stat2 = (diff2 * diff2) / (b2 + c2)
    p2 = 1 - chi2.cdf(stat2, df=1)
else:
    p2 = 1.0
print(f"a={a2}, b={b2}, c={c2}, d={d2}, McNemar p-value: {p2:.4f}")

print("\n=== ИТОГ ===")
print(f"Разница пул=50 vs пул=10: {recall50-recall10:+.4f} ({(recall50-recall10)*100:+.2f} п.п.), p={p_exact:.4f} (n={n})")
if p_exact < 0.05:
    print("Значимо -- пул=50 статистически лучше пул=10.")
else:
    print("Не значимо даже на увеличенной выборке -- разница неотличима от шума, пул=50 обоснован только прецедентом статьи, не собственной значимостью.")

In [ ]:
def hybrid_top100(question):
    q_emb = vo.embed([question], model="voyage-4", input_type="query").embeddings[0]
    pipeline = [
        {"$rankFusion": {
            "input": {"pipelines": {
                "vectorPipeline": [
                    {"$vectorSearch": {"index": "vector_index_full", "path": "embedding_voyage",
                                        "queryVector": q_emb, "numCandidates": 400, "limit": 100}}
                ],
                "fullTextPipeline": [
                    {"$search": {"index": "text_index_full", "text": {"query": question, "path": "context"}}},
                    {"$limit": 100},
                ],
            }},
            "combination": {"weights": {"vectorPipeline": 0.5, "fullTextPipeline": 0.5}},
        }},
        {"$limit": 100},
        {"$project": {"context_id": 1, "context": 1, "_id": 0}},
    ]
    return list(coll.aggregate(pipeline))

save_path_candidates100 = f"{out_dir}/candidates_top100_900.json"
if os.path.exists(save_path_candidates100):
    with open(save_path_candidates100) as f:
        candidates100 = json.load(f)
    print(f"Кандидаты загружены с диска: {len(candidates100)}")
else:
    candidates100 = {}

t0 = time.time()
count = 0
for i, row in eval_set.iterrows():
    q = row["question"]
    if q not in candidates100:
        candidates100[q] = hybrid_top100(q)
    count += 1
    if count % 50 == 0:
        with open(save_path_candidates100, "w") as f:
            json.dump(candidates100, f)
        elapsed = int(time.time() - t0)
        print(f"  кандидаты100: {count}/{len(eval_set)}, {elapsed} сек (сохранено)")
with open(save_path_candidates100, "w") as f:
    json.dump(candidates100, f)
print(f"Кандидаты готовы: {len(candidates100)}, за {int(time.time()-t0)} сек")

In [ ]:
def rerank_cohere(question, docs, top_n=10):
    texts = [d["context"] for d in docs]
    resp = co.rerank(model="rerank-v4.0-pro", query=question, documents=texts, top_n=top_n)
    return [docs[r.index] for r in resp.results]

save_path_reranked100 = f"{out_dir}/reranked_pool100_900.json"
if os.path.exists(save_path_reranked100):
    with open(save_path_reranked100) as f:
        reranked100 = json.load(f)
else:
    reranked100 = {}

t0 = time.time()
count = 0
for i, row in eval_set.iterrows():
    q = row["question"]
    if q not in reranked100:
        reranked100[q] = rerank_cohere(q, candidates100[q], top_n=10)
    count += 1
    if count % 50 == 0:
        with open(save_path_reranked100, "w") as f:
            json.dump(reranked100, f)
        elapsed = int(time.time() - t0)
        print(f"  rerank пул=100: {count}/{len(eval_set)}, {elapsed} сек (сохранено)")
with open(save_path_reranked100, "w") as f:
    json.dump(reranked100, f)
print(f"Готово: {len(reranked100)}, за {int(time.time()-t0)} сек")

In [ ]:
def hit_flags(results_dict, eval_df, k=5):
    flags = {}
    for _, row in eval_df.iterrows():
        q = row["question"]
        top_ids = [d["context_id"] for d in results_dict[q][:k]]
        flags[q] = 1 if row["context_id"] in top_ids else 0
    return flags

flags_pool50 = hit_flags(reranked50, eval_set, k=5)
flags_pool100 = hit_flags(reranked100, eval_set, k=5)

questions = eval_set["question"].tolist()
n = len(questions)
recall50 = sum(flags_pool50.values()) / n
recall100 = sum(flags_pool100.values()) / n
print(f"n = {n}")
print(f"Пул=50 Recall@5: {recall50:.4f}")
print(f"Пул=100 Recall@5: {recall100:.4f}")

def exact_mcnemar(flags_a, flags_b, questions):
    a = b = c = d = 0
    for q in questions:
        fa, fb = flags_a[q], flags_b[q]
        if fa == 1 and fb == 1: a += 1
        elif fa == 1 and fb == 0: b += 1
        elif fa == 0 and fb == 1: c += 1
        else: d += 1
    return a, b, c, d

from scipy.stats import binomtest

a, b, c, d = exact_mcnemar(flags_pool50, flags_pool100, questions)
print(f"\na(оба верно)={a}, b(сломано)={b}, c(спасено)={c}, d(оба неверно)={d}")
print(f"Discordant pairs: b+c={b+c} ({(b+c)/n*100:.1f}%)")
if b + c > 0:
    p_exact = binomtest(min(b, c), b + c, 0.5).pvalue
else:
    p_exact = 1.0
print(f"McNemar exact p-value: {p_exact:.4f}")

print(f"\n=== ИТОГ ===")
print(f"Разница пул=100 vs пул=50: {recall100-recall50:+.4f} ({(recall100-recall50)*100:+.2f} п.п.), p={p_exact:.4f} (n={n})")
if p_exact < 0.05:
    print("Значимо -- пул=100 статистически лучше пул=50.")
else:
    print("Не значимо -- пул=50, похоже, уже близко к плато для нашего корпуса, дальнейшее расширение не оправдано.")

In [ ]:
!pwd

In [ ]:
!ls /content


In [ ]:
from google.colab import files
files.view()